# song-search -> Geode port build (Windows x86_64)

Cross-compiles `matcool.song-search` for **Geometry Dash 2.2081 (x86_64 / Win64)** on a
Linux kernel, per the Geode docs ("Required C++ Tools" -> Linux) plus matcool's own
`linux-win-cross` gist.

Heavy downloads (xwin splat, Geode SDK, bindings, toolchain) live under `/tmp/work`,
so the kernel output stays small: the mod source, the `.geode`, and `BUILD-LOG.txt`.

1. Install clang / cmake / git / ninja / lld
2. `xwin` splat the MSVC CRT + Windows SDK
3. `matcool/clang-msvc-sdk` CMake toolchain (clang-cl, `x86_64-pc-windows-msvc`)
4. Geode CLI + Geode SDK + 2.2081 bindings
5. Write the mod source, run the pure-C++ logic test natively
6. Configure + build the `.geode`
7. Verify the built DLL architecture and the `mod.json`

CPU only, no GPU/TPU.

In [ ]:
%%bash

set -uo pipefail
mkdir -p /tmp/work
LOG=/kaggle/working/BUILD-LOG.txt
: > "$LOG"
log(){ echo "== $* ==" | tee -a "$LOG"; }

# --- deps as the Geode docs "Required C++ Tools -> Linux" describe ---
log "1. deps: clang clang-tools lld llvm cmake git"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq >/dev/null 2>&1 || echo "apt update failed" >> "$LOG"
apt-get install -y -qq --no-install-recommends \
    git cmake ninja-build clang clang-tools lld llvm libssl3 \
    unzip xz-utils zip ca-certificates curl wget gpg >/dev/null 2>&1 \
  || echo "apt install partially failed" >> "$LOG"

# The docs note that some distros ship clang cmake that is too old: Geode
# needs Clang 19+ (21+ recommended) and CMake 3.29+, so pull both from the
# upstream repos when the default is too old.
. /etc/os-release
CODENAME=${VERSION_CODENAME:-noble}

CLANG_VER=$(clang --version | grep -oP 'version [0-9]+' | head -1 | grep -oP '[0-9]+')
if [ -z "$CLANG_VER" ] || [ "$CLANG_VER" -lt 19 ]; then
  log "clang ${CLANG_VER:-none} < 19 -> apt.llvm.org llvm-toolchain-$CODENAME-19"
  wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/llvm.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/llvm.gpg] https://apt.llvm.org/${CODENAME}/ llvm-toolchain-${CODENAME}-19 main" \
    > /etc/apt/sources.list.d/llvm19.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq --no-install-recommends clang-19 lld-19 llvm-19 \
    >/dev/null 2>&1 || echo "apt.llvm.org clang-19 failed" >> "$LOG"
  export PATH=/usr/lib/llvm-19/bin:$PATH
  ln -sf /usr/lib/llvm-19/bin/* /usr/local/bin/ 2>/dev/null || true
  hash -r
fi

CMAKE_VER=$(cmake --version | head -1 | grep -oP '[0-9]+\.[0-9]+' | head -1)
if [ -n "$CMAKE_VER" ] && awk "BEGIN{exit !($CMAKE_VER < 3.29)}"; then
  log "cmake $CMAKE_VER < 3.29 -> apt.kitware.com"
  wget -qO- https://apt.kitware.com/keys/kitware-archive-latest.asc 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/kitware.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/kitware.gpg] https://apt.kitware.com/ubuntu ${CODENAME} main" \
    > /etc/apt/sources.list.d/kitware.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq cmake >/dev/null 2>&1 || true
  hash -r
fi

{
  echo "cmake : $(cmake --version | head -1)"
  echo "git   : $(git --version)"
  echo "ninja : $(ninja --version)"
  echo "clang : $(clang --version | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

# Debian/Ubuntu never ship the clang-cl binary, and upstream dropped the
# x86_64 linux tarballs after 18.x. clang selects the CL driver purely from
# argv[0], so a link to clang named "clang-cl" is equivalent.
CLANG_DIR=$(dirname "$(command -v clang)")
if ! command -v clang-cl >/dev/null; then
  log "clang-cl absent -> linking $CLANG_DIR/clang as clang-cl"
  ln -sf "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl" 2>/dev/null \
    || cp -f "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl"
  ln -sf "$CLANG_DIR/clang" /usr/local/bin/clang-cl 2>/dev/null || true
  hash -r
fi

{
  echo "clang : $(clang --version | head -1)"
  echo "clang-cl: $(clang-cl --version 2>&1 | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

command -v clang-cl >/dev/null || { echo "FATAL clang-cl missing" | tee -a "$LOG"; exit 2; }
command -v lld-link >/dev/null || { echo "FATAL lld-link missing" | tee -a "$LOG"; exit 2; }
log "DEPS DONE"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "2. xwin splat (MSVC CRT + Windows SDK)"

XWIN_VER=0.10.0
if [ ! -x /tmp/work/xwin/xwin ]; then
  curl -sSL -o /tmp/work/xwin.tar.gz \
    "https://github.com/Jake-Shadle/xwin/releases/download/${XWIN_VER}/xwin-${XWIN_VER}-x86_64-unknown-linux-musl.tar.gz" \
    || { echo "xwin download failed" | tee -a "$LOG"; exit 3; }
  mkdir -p /tmp/work/xwin
  tar -xzf /tmp/work/xwin.tar.gz -C /tmp/work/xwin --strip-components=1
  chmod +x /tmp/work/xwin/xwin
fi
export PATH=/tmp/work/xwin:$PATH
xwin --version | tee -a "$LOG"

# xwin's cache skips already-downloaded payloads, so retrying accumulates
# progress through a flaky connection. Default --arch is x86_64 and the
# default --variant is desktop; debug libs are excluded (Release build).
export XWIN_ACCEPT_LICENSE=1
DONE=0
for attempt in $(seq 1 60); do
  echo "--- xwin splat attempt $attempt/60 ---" >> "$LOG"
  # NB: --accept-license/--arch/--variant/--timeout/--cache-dir are GLOBAL
  # (before the subcommand); --output belongs to the splat subcommand.
  xwin --accept-license --arch x86_64 --variant desktop \
       --timeout 300 --cache-dir /tmp/work/xwin-cache \
       splat --include-debug-libs --output /tmp/work/splat >>"$LOG" 2>&1
  RC=$?
  if [ $RC -eq 0 ]; then
    DONE=1
    log "xwin splat OK on attempt $attempt"
    break
  fi
  echo "attempt $attempt rc=$RC" >> "$LOG"
  sleep 3
done
echo "xwin done=$DONE" | tee -a "$LOG"

echo "--- splat layout ---" | tee -a "$LOG"
du -sh /tmp/work/splat 2>>"$LOG" || true
for d in crt/include crt/lib/x86_64 \
         sdk/include/ucrt sdk/include/um sdk/include/shared sdk/include/winrt \
         sdk/lib/ucrt/x86_64 sdk/lib/um/x86_64; do
  if [ -d "/tmp/work/splat/$d" ]; then
    echo "  OK   $d" | tee -a "$LOG"
  else
    echo "  MISS $d" | tee -a "$LOG"
  fi
done
ls /tmp/work/splat/sdk/include/um/Windows.h >/dev/null 2>&1 \
  && echo "  Windows.h present" | tee -a "$LOG" \
  || echo "  Windows.h MISSING" | tee -a "$LOG"
[ "$DONE" = "1" ] || exit 4

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "3. toolchain + Geode CLI + Geode SDK + bindings"
mkdir -p /tmp/work && cd /tmp/work || exit 5
export PATH=/usr/local/bin:$PATH

if [ ! -d toolchain/.git ]; then
  git clone --depth 1 https://github.com/matcool/clang-msvc-sdk.git toolchain >>"$LOG" 2>&1 \
    || { echo "toolchain clone failed" | tee -a "$LOG"; exit 6; }
fi
ls toolchain | tee -a "$LOG"

GEODE_CLI_VER=v3.9.0
if [ ! -x /tmp/work/geode-cli/geode ]; then
  curl -sSL -o /tmp/work/geode.zip \
    "https://github.com/geode-sdk/cli/releases/download/${GEODE_CLI_VER}/geode-cli-${GEODE_CLI_VER}-linux.zip" \
    || { echo "geode cli download failed" | tee -a "$LOG"; exit 7; }
  rm -rf /tmp/work/geode-cli
  unzip -q -o /tmp/work/geode.zip -d /tmp/work/geode-cli
  chmod +x /tmp/work/geode-cli/geode
fi
export PATH=/tmp/work/geode-cli:$PATH
geode --version | tee -a "$LOG"

# `geode project check` (run by the SDK's CMake) needs a current CLI profile.
# Configure a throwaway one pointing at a scratch dir - we only cross-build
# here, never run the game. It also wants the usual Geode install layout
# present so it can read the list of installed mods.
mkdir -p /tmp/work/gd-scratch/geode/mods
mkdir -p /tmp/work/gd-scratch/geode/config
mkdir -p /tmp/work/gd-scratch/geode/logs
mkdir -p /tmp/work/gd-scratch/geode/unzipped
if ! geode profile list 2>/dev/null | grep -q kaggle; then
  geode profile add --name kaggle /tmp/work/gd-scratch >>"$LOG" 2>&1 || true
fi
geode profile switch kaggle >>"$LOG" 2>&1 || true
echo "profiles:" | tee -a "$LOG"
geode profile list 2>&1 | tee -a "$LOG"

if [ ! -d sdk/.git ]; then
  git clone --depth 1 --branch main https://github.com/geode-sdk/sdk.git sdk >>"$LOG" 2>&1 \
    || { echo "geode sdk clone failed" | tee -a "$LOG"; exit 8; }
fi
geode sdk set-path "/tmp/work/sdk" >>"$LOG" 2>&1 || true
geode sdk version | tee -a "$LOG"
ls sdk/loader/include/link/win64 | tee -a "$LOG"

# The SDK links the Geode loader import library from <sdk>/bin/<version>/.
# `geode sdk install-binaries` fetches it; do it manually if the CLI cannot.
SDK_VER=$(cat sdk/VERSION 2>/dev/null | tr -d '[:space:]')
echo "sdk version: $SDK_VER" | tee -a "$LOG"
if [ -n "$SDK_VER" ] && [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  geode sdk install-binaries >>"$LOG" 2>&1 || true
fi
if [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  log "install-binaries did not provide Geode.lib -> fetching loader zip directly"
  for attempt in $(seq 1 20); do
    curl -sSL -o /tmp/work/geode-win.zip \
      "https://github.com/geode-sdk/geode/releases/download/v${SDK_VER}/geode-v${SDK_VER}-win.zip" && break
    echo "loader zip attempt $attempt failed" >> "$LOG"
    sleep 5
  done
  rm -rf /tmp/work/geode-win && mkdir -p /tmp/work/geode-win
  unzip -q -o /tmp/work/geode-win.zip -d /tmp/work/geode-win || true
  mkdir -p "sdk/bin/${SDK_VER}"
  FOUND=$(find /tmp/work/geode-win -name 'Geode.lib' | head -1)
  if [ -n "$FOUND" ]; then
    cp -f "$FOUND" "sdk/bin/${SDK_VER}/Geode.lib"
    log "installed Geode.lib from $FOUND"
  else
    echo "WARNING: Geode.lib not found in loader zip" | tee -a "$LOG"
    find /tmp/work/geode-win -maxdepth 3 -type f | head -30 | tee -a "$LOG"
  fi
fi
ls -la "sdk/bin/${SDK_VER}/" 2>&1 | tee -a "$LOG"

# Geode generates binding headers at configure time from the bindings repo;
# pre-seed a clone so configure does not fetch it itself
if [ ! -d bindings/.git ]; then
  git clone --depth 1 https://github.com/geode-sdk/bindings.git bindings >>"$LOG" 2>&1 \
    || { echo "bindings clone failed" | tee -a "$LOG"; exit 9; }
fi
ls bindings/bindings/2.2081 | tee -a "$LOG"
log "TOOLCHAIN DONE"

In [ ]:
import base64, json, pathlib
FILES = {
 "mod.json": "ewoJImdlb2RlIjogIjUuMTEuMSIsCgkiZ2QiOiB7CgkJIndpbiI6ICIqIiwKCQkiYW5kcm9pZCI6ICIqIiwKCQkibWFjIjogIioiLAoJCSJpb3MiOiAiKiIKCX0sCgkiaWQiOiAibWF0Y29vbC5zb25nLXNlYXJjaCIsCgkibmFtZSI6ICJTb25nIFNlYXJjaCIsCgkidmVyc2lvbiI6ICJ2MC4yLjgiLAoJImRldmVsb3BlcnMiOiBbCgkJIm5vdGxvdXN5Ym9vayIsCgkJIm1hdGNvb2wiCgldLAoJImRlc2NyaXB0aW9uIjogIlNlYXJjaCB5b3VyIGRvd25sb2FkZWQgc29uZ3Mgd2l0aCBmdXp6eSBtYXRjaGluZywgcGx1cyBKdWtlYm94IE5PTkcgc3VwcG9ydC4iLAoJInRhZ3MiOiBbCgkJIm11c2ljIiwKCQkiaW50ZXJmYWNlIiwKCQkiZW5oYW5jZW1lbnQiCgldLAoJInNldHRpbmdzIjogewoJCSJnZW5lcmFsIjogewoJCQkidHlwZSI6ICJ0aXRsZSIsCgkJCSJuYW1lIjogIkdlbmVyYWwiCgkJfSwKCQkiZGlzYWJsZS1tb2QiOiB7CgkJCSJ0eXBlIjogImJvb2wiLAoJCQkibmFtZSI6ICJEaXNhYmxlIFNvbmcgU2VhcmNoIiwKCQkJImRlc2NyaXB0aW9uIjogIlR1cm4gdGhlIHNlYXJjaCBkaWFsb2cgb2ZmIGVudGlyZWx5IGFuZCByZXN0b3JlIHRoZSBkZWZhdWx0IHNvbmcgYnJvd3Nlci4iLAoJCQkiZGVmYXVsdCI6IGZhbHNlCgkJfSwKCQkianVrZWJveCI6IHsKCQkJInR5cGUiOiAidGl0bGUiLAoJCQkibmFtZSI6ICJKdWtlYm94IE5PTkcgc3VwcG9ydCIKCQl9LAoJCSJub25nLXN5bWJvbHMiOiB7CgkJCSJ0eXBlIjogImJvb2wiLAoJCQkibmFtZSI6ICJTaG93IE5PTkcgc3ltYm9scyIsCgkJCSJkZXNjcmlwdGlvbiI6ICJEcmF3IEp1a2Vib3gncyBwaW4gZGlzYyBvbiBlYWNoIHNvbmcgcm93J3MgdG9wLWxlZnQgY29ybmVyOyBjbGlja2luZyBpdCBvcGVucyBKdWtlYm94J3Mgbm9uZyBwaWNrZXIgZm9yIHRoYXQgc29uZy4gUmVxdWlyZXMgSnVrZWJveCB0byBiZSBpbnN0YWxsZWQ7IGRvZXMgbm90aGluZyB3aXRob3V0IGl0LiIsCgkJCSJkZWZhdWx0IjogdHJ1ZQoJCX0KCX0sCgkicmVzb3VyY2VzIjogewoJCSJzcHJpdGVzaGVldHMiOiB7CgkJCSJTb25nU2VhcmNoU2hlZXQiOiBbCgkJCQkicmVzb3VyY2VzLyoucG5nIgoJCQldCgkJfQoJfSwKCSJsaW5rcyI6IHsKCQkic291cmNlIjogImh0dHBzOi8vZ2l0aHViLmNvbS9ub3Rsb3VzeWJvb2svc29uZy1zZWFyY2gtZ2VvZGUiCgl9LAoJImlzc3VlcyI6IHsKCQkiaW5mbyI6ICJSZXBvcnQgaXNzdWVzIG9uIHRoZSBHaXRIdWIgcmVwb3NpdG9yeSBvZiB0aGlzIHBvcnQuIiwKCQkidXJsIjogImh0dHBzOi8vZ2l0aHViLmNvbS9ub3Rsb3VzeWJvb2svc29uZy1zZWFyY2gtZ2VvZGUvaXNzdWVzIgoJfQp9Cg==",
 "CMakeLists.txt": "Y21ha2VfbWluaW11bV9yZXF1aXJlZChWRVJTSU9OIDMuMjEpCnNldChDTUFLRV9DWFhfU1RBTkRBUkQgMjMpCnNldChDTUFLRV9DWFhfU1RBTkRBUkRfUkVRVUlSRUQgT04pCnNldChDTUFLRV9DWFhfVklTSUJJTElUWV9QUkVTRVQgaGlkZGVuKQoKcHJvamVjdChzb25nLXNlYXJjaCBWRVJTSU9OIDEuMC4wKQoKIyBBZGQgYWxsIHNvdXJjZSBmaWxlcyBpbnNpZGUgc3JjIChyZWN1cnNpdmVseSkKZmlsZShHTE9CX1JFQ1VSU0UgU09VUkNFUyBDT05GSUdVUkVfREVQRU5EUyBzcmMvKi5jcHApCgojIFNldCB1cCB0aGUgbW9kIGJpbmFyeQphZGRfbGlicmFyeSgke1BST0pFQ1RfTkFNRX0gU0hBUkVEICR7U09VUkNFU30pCgppZiAoTk9UIERFRklORUQgRU5We0dFT0RFX1NES30pCiAgICBtZXNzYWdlKEZBVEFMX0VSUk9SICJVbmFibGUgdG8gZmluZCBHZW9kZSBTREshIFBsZWFzZSBkZWZpbmUgR0VPREVfU0RLIGVudmlyb25tZW50IHZhcmlhYmxlIHRvIHBvaW50IHRvIEdlb2RlIikKZWxzZSgpCiAgICBtZXNzYWdlKFNUQVRVUyAiRm91bmQgR2VvZGU6ICRFTlZ7R0VPREVfU0RLfSIpCmVuZGlmKCkKCmFkZF9zdWJkaXJlY3RvcnkoJEVOVntHRU9ERV9TREt9ICR7Q01BS0VfQ1VSUkVOVF9CSU5BUllfRElSfS9nZW9kZSkKCiMgU2V0IHVwIGRlcGVuZGVuY2llcywgcmVzb3VyY2VzLCBhbmQgbGluayBHZW9kZS4Kc2V0dXBfZ2VvZGVfbW9kKCR7UFJPSkVDVF9OQU1FfSkK",
 ".gitignore": "YnVpbGQvCi50b29scy8KKi5nZW9kZQoKIyBLYWdnbGUgYnVpbGQgc2NyYXRjaCBvdXRwdXQgKGxvZ3MgLyBmZXRjaGVkIHBhY2thZ2VzKQoua2FnZ2xlLyovb3V0Ki8KLmthZ2dsZS8qL28qLwoua2FnZ2xlLyovcGtnLwoua2FnZ2xlLyovcHVsbGVkKi8KCiMgS2lsby1pbnN0YWxsZWQgc2tpbGwgY29weSAoc291cmNlIG9mIHRydXRoIGlzIC5rYWdnbGUvc2tpbGxzLykKLmtpbG9jb2RlLwo=",
 "src/main.cpp": "...",
 "src/fts_fuzzy_match.h": "Ly8gTElDRU5TRQovLwovLyAgIFRoaXMgc29mdHdhcmUgaXMgZHVhbC1saWNlbnNlZCB0byB0aGUgcHVibGljIGRvbWFpbiBhbmQgdW5kZXIgdGhlIGZvbGxvd2luZwovLyAgIGxpY2Vuc2U6IHlvdSBhcmUgZ3JhbnRlZCBhIHBlcnBldHVhbCwgaXJyZXZvY2FibGUgbGljZW5zZSB0byBjb3B5LCBtb2RpZnksCi8vICAgcHVibGlzaCwgYW5kIGRpc3RyaWJ1dGUgdGhpcyBmaWxlIGFzIHlvdSBzZWUgZml0LgovLwovLyBWRVJTSU9OCi8vICAgMC4yLjAgICgyMDE3LTAyLTE4KSAgU2NvcmVkIG1hdGNoZXMgcGVyZm9ybSBleGhhdXN0aXZlIHNlYXJjaCBmb3IgYmVzdCBzY29yZQovLyAgIDAuMS4wICAoMjAxNi0wMy0yOCkgIEluaXRpYWwgcmVsZWFzZQovLwovLyBBVVRIT1IKLy8gICBGb3JyZXN0IFNtaXRoCi8vCi8vIE5PVEVTCi8vICAgQ29tcGlsaW5nCi8vICAgICBZb3UgTVVTVCBhZGQgJyNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OJyBiZWZvcmUgaW5jbHVkaW5nIHRoaXMgaGVhZGVyIGluIE9ORSBzb3VyY2UgZmlsZSB0byBjcmVhdGUgaW1wbGVtZW50YXRpb24uCi8vCi8vICAgZnV6enlfbWF0Y2hfc2ltcGxlKC4uLikKLy8gICAgIFJldHVybnMgdHJ1ZSBpZiBlYWNoIGNoYXJhY3RlciBpbiBwYXR0ZXJuIGlzIGZvdW5kIHNlcXVlbnRpYWxseSB3aXRoaW4gc3RyCi8vCi8vICAgZnV6enlfbWF0Y2goLi4uKQovLyAgICAgUmV0dXJucyB0cnVlIGlmIHBhdHRlcm4gaXMgZm91bmQgQU5EIGNhbGN1bGF0ZXMgYSBzY29yZS4KLy8gICAgIFBlcmZvcm1zIGV4aGF1c3RpdmUgc2VhcmNoIHZpYSByZWN1cnNpb24gdG8gZmluZCBhbGwgcG9zc2libGUgbWF0Y2hlcyBhbmQgbWF0Y2ggd2l0aCBoaWdoZXN0IHNjb3JlLgovLyAgICAgU2NvcmVzIHZhbHVlcyBoYXZlIG5vIGludHJpbnNpYyBtZWFuaW5nLiBQb3NzaWJsZSBzY29yZSByYW5nZSBpcyBub3Qgbm9ybWFsaXplZCBhbmQgdmFyaWVzIHdpdGggcGF0dGVybi4KLy8gICAgIFJlY3Vyc2lvbiBpcyBsaW1pdGVkIGludGVybmFsbHkgKGRlZmF1bHQ9MTApIHRvIHByZXZlbnQgZGVnZW5lcmF0ZSBjYXNlcyAocGF0dGVybj0iYWFhYWFhIiBzdHI9ImFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYSIpCi8vICAgICBVc2VzIHVpbnQ4X3QgZm9yIG1hdGNoIGluZGljZXMuIFRoZXJlZm9yZSBwYXR0ZXJucyBhcmUgbGltaXRlZCB0byAyNTYgY2hhcmFjdGVycy4KLy8gICAgIFNjb3JlIHN5c3RlbSBzaG91bGQgYmUgdHVuZWQgZm9yIFlPVVIgdXNlIGNhc2UuIFdvcmRzLCBzZW50ZW5jZXMsIGZpbGUgbmFtZXMsIG9yIG1ldGhvZCBuYW1lcyBhbGwgcHJlZmVyIGRpZmZlcmVudCB0dW5pbmcuCgojaWZuZGVmIEZUU19GVVpaWV9NQVRDSF9ICiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0gKCiNpbmNsdWRlIDxjc3RkaW50PiAvLyB1aW50OF90CiNpbmNsdWRlIDxjdHlwZS5oPiAvLyA6OnRvbG93ZXIsIDo6dG91cHBlcgojaW5jbHVkZSA8Y3N0cmluZz4gLy8gbWVtY3B5CgojaW5jbHVkZSA8Y3N0ZGlvPgoKLy8gUHVibGljIGludGVyZmFjZQpuYW1lc3BhY2UgZnRzIHsKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cik7CiAgICBzdGF0aWMgYm9vbCBmdXp6eV9tYXRjaChjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0ciwgaW50ICYgb3V0U2NvcmUpOwogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlLCB1aW50OF90ICogbWF0Y2hlcywgaW50IG1heE1hdGNoZXMpOwp9CgojaWZkZWYgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCm5hbWVzcGFjZSBmdHMgewoKICAgIC8vIEZvcndhcmQgZGVjbGFyYXRpb25zIGZvciAicHJpdmF0ZSIgaW1wbGVtZW50YXRpb24KICAgIG5hbWVzcGFjZSBmdXp6eV9pbnRlcm5hbCB7CiAgICAgICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2hfcmVjdXJzaXZlKGNvbnN0IGNoYXIgKiBwYXR0ZXJuLCBjb25zdCBjaGFyICogc3RyLCBpbnQgJiBvdXRTY29yZSwgY29uc3QgY2hhciAqIHN0ckJlZ2luLAogICAgICAgICAgICB1aW50OF90IGNvbnN0ICogc3JjTWF0Y2hlcywgIHVpbnQ4X3QgKiBuZXdNYXRjaGVzLCAgaW50IG1heE1hdGNoZXMsIGludCBuZXh0TWF0Y2gsCiAgICAgICAgICAgIGludCAmIHJlY3Vyc2lvbkNvdW50LCBpbnQgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFB1YmxpYyBpbnRlcmZhY2UKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cikgewogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgIHsKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpCiAgICAgICAgICAgICAgICArK3BhdHRlcm47CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgcmV0dXJuICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CiAgICB9CgogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlKSB7CgogICAgICAgIHVpbnQ4X3QgbWF0Y2hlc1syNTZdOwogICAgICAgIHJldHVybiBmdXp6eV9tYXRjaChwYXR0ZXJuLCBzdHIsIG91dFNjb3JlLCBtYXRjaGVzLCBzaXplb2YobWF0Y2hlcykpOwogICAgfQoKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoKGNoYXIgY29uc3QgKiBwYXR0ZXJuLCBjaGFyIGNvbnN0ICogc3RyLCBpbnQgJiBvdXRTY29yZSwgdWludDhfdCAqIG1hdGNoZXMsIGludCBtYXhNYXRjaGVzKSB7CiAgICAgICAgaW50IHJlY3Vyc2lvbkNvdW50ID0gMDsKICAgICAgICBpbnQgcmVjdXJzaW9uTGltaXQgPSAxMDsKCiAgICAgICAgcmV0dXJuIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyLCBvdXRTY29yZSwgc3RyLCBudWxscHRyLCBtYXRjaGVzLCBtYXhNYXRjaGVzLCAwLCByZWN1cnNpb25Db3VudCwgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFByaXZhdGUgaW1wbGVtZW50YXRpb24KICAgIHN0YXRpYyBib29sIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUoY29uc3QgY2hhciAqIHBhdHRlcm4sIGNvbnN0IGNoYXIgKiBzdHIsIGludCAmIG91dFNjb3JlLAogICAgICAgIGNvbnN0IGNoYXIgKiBzdHJCZWdpbiwgdWludDhfdCBjb25zdCAqIHNyY01hdGNoZXMsIHVpbnQ4X3QgKiBtYXRjaGVzLCBpbnQgbWF4TWF0Y2hlcywKICAgICAgICBpbnQgbmV4dE1hdGNoLCBpbnQgJiByZWN1cnNpb25Db3VudCwgaW50IHJlY3Vyc2lvbkxpbWl0KQogICAgewogICAgICAgIC8vIENvdW50IHJlY3Vyc2lvbnMKICAgICAgICArK3JlY3Vyc2lvbkNvdW50OwogICAgICAgIGlmIChyZWN1cnNpb25Db3VudCA+PSByZWN1cnNpb25MaW1pdCkKICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAvLyBEZXRlY3QgZW5kIG9mIHN0cmluZ3MKICAgICAgICBpZiAoKnBhdHRlcm4gPT0gJ1wwJyB8fCAqc3RyID09ICdcMCcpCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKCiAgICAgICAgLy8gUmVjdXJzaW9uIHBhcmFtcwogICAgICAgIGJvb2wgcmVjdXJzaXZlTWF0Y2ggPSBmYWxzZTsKICAgICAgICB1aW50OF90IGJlc3RSZWN1cnNpdmVNYXRjaGVzWzI1Nl07CiAgICAgICAgaW50IGJlc3RSZWN1cnNpdmVTY29yZSA9IDA7CgogICAgICAgIC8vIExvb3AgdGhyb3VnaCBwYXR0ZXJuIGFuZCBzdHIgbG9va2luZyBmb3IgYSBtYXRjaAogICAgICAgIGJvb2wgZmlyc3RfbWF0Y2ggPSB0cnVlOwogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgewoKICAgICAgICAgICAgLy8gRm91bmQgbWF0Y2gKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpIHsKCiAgICAgICAgICAgICAgICAvLyBTdXBwbGllZCBtYXRjaGVzIGJ1ZmZlciB3YXMgdG9vIHNob3J0CiAgICAgICAgICAgICAgICBpZiAobmV4dE1hdGNoID49IG1heE1hdGNoZXMpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAgICAgICAgIC8vICJDb3B5LW9uLVdyaXRlIiBzcmNNYXRjaGVzIGludG8gbWF0Y2hlcwogICAgICAgICAgICAgICAgaWYgKGZpcnN0X21hdGNoICYmIHNyY01hdGNoZXMpIHsKICAgICAgICAgICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgc3JjTWF0Y2hlcywgbmV4dE1hdGNoKTsKICAgICAgICAgICAgICAgICAgICBmaXJzdF9tYXRjaCA9IGZhbHNlOwogICAgICAgICAgICAgICAgfQoKICAgICAgICAgICAgICAgIC8vIFJlY3Vyc2l2ZSBjYWxsIHRoYXQgInNraXBzIiB0aGlzIG1hdGNoCiAgICAgICAgICAgICAgICB1aW50OF90IHJlY3Vyc2l2ZU1hdGNoZXNbMjU2XTsKICAgICAgICAgICAgICAgIGludCByZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgICAgIGlmIChmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyICsgMSwgcmVjdXJzaXZlU2NvcmUsIHN0ckJlZ2luLCBtYXRjaGVzLCByZWN1cnNpdmVNYXRjaGVzLCBzaXplb2YocmVjdXJzaXZlTWF0Y2hlcyksIG5leHRNYXRjaCwgcmVjdXJzaW9uQ291bnQsIHJlY3Vyc2lvbkxpbWl0KSkgewoKICAgICAgICAgICAgICAgICAgICAvLyBQaWNrIGJlc3QgcmVjdXJzaXZlIHNjb3JlCiAgICAgICAgICAgICAgICAgICAgaWYgKCFyZWN1cnNpdmVNYXRjaCB8fCByZWN1cnNpdmVTY29yZSA+IGJlc3RSZWN1cnNpdmVTY29yZSkgewogICAgICAgICAgICAgICAgICAgICAgICBtZW1jcHkoYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIHJlY3Vyc2l2ZU1hdGNoZXMsIDI1Nik7CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RSZWN1cnNpdmVTY29yZSA9IHJlY3Vyc2l2ZVNjb3JlOwogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICByZWN1cnNpdmVNYXRjaCA9IHRydWU7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQWR2YW5jZQogICAgICAgICAgICAgICAgbWF0Y2hlc1tuZXh0TWF0Y2grK10gPSAodWludDhfdCkoc3RyIC0gc3RyQmVnaW4pOwogICAgICAgICAgICAgICAgKytwYXR0ZXJuOwogICAgICAgICAgICB9CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgLy8gRGV0ZXJtaW5lIGlmIGZ1bGwgcGF0dGVybiB3YXMgbWF0Y2hlZAogICAgICAgIGJvb2wgbWF0Y2hlZCA9ICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CgogICAgICAgIC8vIENhbGN1bGF0ZSBzY29yZQogICAgICAgIGlmIChtYXRjaGVkKSB7CiAgICAgICAgICAgIGNvbnN0IGludCBzZXF1ZW50aWFsX2JvbnVzID0gMTU7ICAgICAgICAgICAgLy8gYm9udXMgZm9yIGFkamFjZW50IG1hdGNoZXMKICAgICAgICAgICAgY29uc3QgaW50IHNlcGFyYXRvcl9ib251cyA9IDMwOyAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBvY2N1cnMgYWZ0ZXIgYSBzZXBhcmF0b3IKICAgICAgICAgICAgY29uc3QgaW50IGNhbWVsX2JvbnVzID0gMzA7ICAgICAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBpcyB1cHBlcmNhc2UgYW5kIHByZXYgaXMgbG93ZXIKICAgICAgICAgICAgY29uc3QgaW50IGZpcnN0X2xldHRlcl9ib251cyA9IDE1OyAgICAgICAgICAvLyBib251cyBpZiB0aGUgZmlyc3QgbGV0dGVyIGlzIG1hdGNoZWQKCiAgICAgICAgICAgIGNvbnN0IGludCBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTU7ICAgICAgLy8gcGVuYWx0eSBhcHBsaWVkIGZvciBldmVyeSBsZXR0ZXIgaW4gc3RyIGJlZm9yZSB0aGUgZmlyc3QgbWF0Y2gKICAgICAgICAgICAgY29uc3QgaW50IG1heF9sZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTE1OyAvLyBtYXhpbXVtIHBlbmFsdHkgZm9yIGxlYWRpbmcgbGV0dGVycwogICAgICAgICAgICBjb25zdCBpbnQgdW5tYXRjaGVkX2xldHRlcl9wZW5hbHR5ID0gLTE7ICAgIC8vIHBlbmFsdHkgZm9yIGV2ZXJ5IGxldHRlciB0aGF0IGRvZXNuJ3QgbWF0dGVyCgogICAgICAgICAgICAvLyBJdGVyYXRlIHN0ciB0byBlbmQKICAgICAgICAgICAgd2hpbGUgKCpzdHIgIT0gJ1wwJykKICAgICAgICAgICAgICAgICsrc3RyOwoKICAgICAgICAgICAgLy8gSW5pdGlhbGl6ZSBzY29yZQogICAgICAgICAgICBvdXRTY29yZSA9IDEwMDsKCiAgICAgICAgICAgIC8vIEFwcGx5IGxlYWRpbmcgbGV0dGVyIHBlbmFsdHkKICAgICAgICAgICAgaW50IHBlbmFsdHkgPSBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ICogbWF0Y2hlc1swXTsKICAgICAgICAgICAgaWYgKHBlbmFsdHkgPCBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eSkKICAgICAgICAgICAgICAgIHBlbmFsdHkgPSBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eTsKICAgICAgICAgICAgb3V0U2NvcmUgKz0gcGVuYWx0eTsKCiAgICAgICAgICAgIC8vIEFwcGx5IHVubWF0Y2hlZCBwZW5hbHR5CiAgICAgICAgICAgIGludCB1bm1hdGNoZWQgPSAoaW50KShzdHIgLSBzdHJCZWdpbikgLSBuZXh0TWF0Y2g7CiAgICAgICAgICAgIG91dFNjb3JlICs9IHVubWF0Y2hlZF9sZXR0ZXJfcGVuYWx0eSAqIHVubWF0Y2hlZDsKCiAgICAgICAgICAgIC8vIEFwcGx5IG9yZGVyaW5nIGJvbnVzZXMKICAgICAgICAgICAgZm9yIChpbnQgaSA9IDA7IGkgPCBuZXh0TWF0Y2g7ICsraSkgewogICAgICAgICAgICAgICAgdWludDhfdCBjdXJySWR4ID0gbWF0Y2hlc1tpXTsKCiAgICAgICAgICAgICAgICBpZiAoaSA+IDApIHsKICAgICAgICAgICAgICAgICAgICB1aW50OF90IHByZXZJZHggPSBtYXRjaGVzW2kgLSAxXTsKCiAgICAgICAgICAgICAgICAgICAgLy8gU2VxdWVudGlhbAogICAgICAgICAgICAgICAgICAgIGlmIChjdXJySWR4ID09IChwcmV2SWR4ICsgMSkpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcXVlbnRpYWxfYm9udXM7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQ2hlY2sgZm9yIGJvbnVzZXMgYmFzZWQgb24gbmVpZ2hib3IgY2hhcmFjdGVyIHZhbHVlCiAgICAgICAgICAgICAgICBpZiAoY3VycklkeCA+IDApIHsKICAgICAgICAgICAgICAgICAgICAvLyBDYW1lbCBjYXNlCiAgICAgICAgICAgICAgICAgICAgY2hhciBuZWlnaGJvciA9IHN0ckJlZ2luW2N1cnJJZHggLSAxXTsKICAgICAgICAgICAgICAgICAgICBjaGFyIGN1cnIgPSBzdHJCZWdpbltjdXJySWR4XTsKICAgICAgICAgICAgICAgICAgICBpZiAoOjppc2xvd2VyKG5laWdoYm9yKSAmJiA6OmlzdXBwZXIoY3VycikpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGNhbWVsX2JvbnVzOwoKICAgICAgICAgICAgICAgICAgICAvLyBTZXBhcmF0b3IKICAgICAgICAgICAgICAgICAgICBib29sIG5laWdoYm9yU2VwYXJhdG9yID0gbmVpZ2hib3IgPT0gJ18nIHx8IG5laWdoYm9yID09ICcgJzsKICAgICAgICAgICAgICAgICAgICBpZiAobmVpZ2hib3JTZXBhcmF0b3IpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcGFyYXRvcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGVsc2UgewogICAgICAgICAgICAgICAgICAgIC8vIEZpcnN0IGxldHRlcgogICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGZpcnN0X2xldHRlcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgfQogICAgICAgIH0KCiAgICAgICAgLy8gUmV0dXJuIGJlc3QgcmVzdWx0CiAgICAgICAgaWYgKHJlY3Vyc2l2ZU1hdGNoICYmICghbWF0Y2hlZCB8fCBiZXN0UmVjdXJzaXZlU2NvcmUgPiBvdXRTY29yZSkpIHsKICAgICAgICAgICAgLy8gUmVjdXJzaXZlIHNjb3JlIGlzIGJldHRlciB0aGFuICJ0aGlzIgogICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIG1heE1hdGNoZXMpOwogICAgICAgICAgICBvdXRTY29yZSA9IGJlc3RSZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgcmV0dXJuIHRydWU7CiAgICAgICAgfQogICAgICAgIGVsc2UgaWYgKG1hdGNoZWQpIHsKICAgICAgICAgICAgLy8gInRoaXMiIHNjb3JlIGlzIGJldHRlciB0aGFuIHJlY3Vyc2l2ZQogICAgICAgICAgICByZXR1cm4gdHJ1ZTsKICAgICAgICB9CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIC8vIG5vIG1hdGNoCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKICAgICAgICB9CiAgICB9Cn0gLy8gbmFtZXNwYWNlIGZ0cwoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9JTVBMRU1FTlRBVElPTgoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9ICg==",
 "tests/logic_test.cpp": "Ly8gU3RhbmRhbG9uZSBsb2dpYyB0ZXN0IGZvciB0aGUgc29uZy1zZWFyY2ggcG9ydC4KLy8gTWlycm9ycyB0aGUgZnV6enkgc2VhcmNoICsgbGFiZWwtZml0ICsgc2Nyb2xsLXdpbmRvdyBsb2dpYyBpbiBzcmMvbWFpbi5jcHAKLy8gd2l0aG91dCBHZW9kZSBoZWFkZXJzLCBzbyBpdCBjYW4gcnVuIG9uIHBsYWluIEdDQy4KI2RlZmluZSBGVFNfRlVaWllfTUFUQ0hfSU1QTEVNRU5UQVRJT04KI2luY2x1ZGUgImZ0c19mdXp6eV9tYXRjaC5oIgoKI2luY2x1ZGUgPGFsZ29yaXRobT4KI2luY2x1ZGUgPGNhc3NlcnQ+CiNpbmNsdWRlIDxjbWF0aD4KI2luY2x1ZGUgPGlvc3RyZWFtPgojaW5jbHVkZSA8c3N0cmVhbT4KI2luY2x1ZGUgPHN0cmluZz4KI2luY2x1ZGUgPHZlY3Rvcj4KCnN0cnVjdCBGYWtlU29uZyB7IGludCBpZDsgc3RkOjpzdHJpbmcgbmFtZTsgc3RkOjpzdHJpbmcgYXJ0aXN0OyB9OwoKc3RhdGljIHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxpbnQsIEZha2VTb25nPj4gZG9fc2VhcmNoKAogICAgY29uc3Qgc3RkOjp2ZWN0b3I8c3RkOjpwYWlyPHN0ZDo6c3RyaW5nLCBGYWtlU29uZz4+JiBkb3dubG9hZGVkLAogICAgY29uc3Qgc3RkOjpzdHJpbmcmIGlucHV0KSB7CiAgICBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8aW50LCBGYWtlU29uZz4+IHNvbmdzOwogICAgc29uZ3MucmVzZXJ2ZShkb3dubG9hZGVkLnNpemUoKSk7CiAgICBjb25zdCBib29sIGVtcHR5X2lucHV0ID0gaW5wdXQuZW1wdHkoKTsKICAgIGZvciAoY29uc3QgYXV0byYgW2luZGV4X3N0ciwgc29uZ10gOiBkb3dubG9hZGVkKSB7CiAgICAgICAgaW50IHNjb3JlID0gMDsKICAgICAgICBpZiAoIWVtcHR5X2lucHV0ICYmICFmdHM6OmZ1enp5X21hdGNoKGlucHV0LmNfc3RyKCksIGluZGV4X3N0ci5jX3N0cigpLCBzY29yZSkpIGNvbnRpbnVlOwogICAgICAgIHNvbmdzLnB1c2hfYmFjayh7c2NvcmUsIHNvbmd9KTsKICAgIH0KICAgIGlmICghZW1wdHlfaW5wdXQgJiYgc29uZ3Muc2l6ZSgpID4gMSkgewogICAgICAgIHN0ZDo6c29ydChzb25ncy5iZWdpbigpLCBzb25ncy5lbmQoKSwgWyZdKGNvbnN0IGF1dG8mIGEsIGNvbnN0IGF1dG8mIGIpIHsKICAgICAgICAgICAgcmV0dXJuIGEuZmlyc3QgPiBiLmZpcnN0OwogICAgICAgIH0pOwogICAgfQogICAgcmV0dXJuIHNvbmdzOwp9CgppbnQgbWFpbigpIHsKICAgIHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxzdGQ6OnN0cmluZywgRmFrZVNvbmc+PiBkb3dubG9hZGVkID0gewogICAgICAgIHsiU3RlcmVvIE1hZG5lc3MgRm9yZXZlciIsIHsxLCAiU3RlcmVvIE1hZG5lc3MiLCAiRm9yZXZlciJ9fSwKICAgICAgICB7IkJhY2sgb24gVHJhY2sgREpWSSIsIHsyLCAiQmFjayBvbiBUcmFjayIsICJESlZJIn19LAogICAgICAgIHsiUG9sYXJnZWlzdCBTdGVwIiwgezMsICJQb2xhcmdlaXN0IiwgIlN0ZXAifX0sCiAgICAgICAgeyJEcnkgT3V0IERKVkkiLCB7NCwgIkRyeSBPdXQiLCAiREpWSSJ9fSwKICAgICAgICB7IkJhc2UgQWZ0ZXIgQmFzZSBESlZJIiwgezUsICJCYXNlIEFmdGVyIEJhc2UiLCAiREpWSSJ9fSwKICAgIH07CgogICAgLy8gMS4gZW1wdHkgaW5wdXQgcmV0dXJucyBldmVyeXRoaW5nLCB1bnNvcnRlZAogICAgYXV0byBhbGwgPSBkb19zZWFyY2goZG93bmxvYWRlZCwgIiIpOwogICAgYXNzZXJ0KGFsbC5zaXplKCkgPT0gNSk7CgogICAgLy8gMi4gZXhhY3QtaXNoIHF1ZXJ5IHJhbmtzIHRoZSByaWdodCBzb25nIGZpcnN0CiAgICBhdXRvIGRqdmkgPSBkb19zZWFyY2goZG93bmxvYWRlZCwgImRqdmkiKTsKICAgIGFzc2VydChkanZpLnNpemUoKSA9PSAzKTsKICAgIGZvciAoYXV0byYgW3Njb3JlLCBzXSA6IGRqdmkpICh2b2lkKXNjb3JlLCBhc3NlcnQocy5hcnRpc3QgPT0gIkRKVkkiKTsKCiAgICAvLyAzLiBmdXp6eSBxdWVyeSBtYXRjaGVzIGFjcm9zcyBuYW1lK2FydGlzdCBibG9iCiAgICBhdXRvIHBvbGFyID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICJwb2xyIik7CiAgICBhc3NlcnQocG9sYXIuc2l6ZSgpID09IDEgJiYgcG9sYXJbMF0uc2Vjb25kLmlkID09IDMpOwoKICAgIC8vIDQuIG5vIG1hdGNoIC0+IGVtcHR5CiAgICBhc3NlcnQoZG9fc2VhcmNoKGRvd25sb2FkZWQsICJ6enpxcXEiKS5lbXB0eSgpKTsKCiAgICAvLyA1LiBzY3JvbGwgY2xhbXA6IG1pcnJvcnMgdXBkYXRlX3Njcm9sbF9lbGVtZW50cyBjbGFtcGluZwogICAgY29uc3RleHByIGZsb2F0IGhlaWdodCA9IDU1LmY7IC8vIHdpZGdldF9oZWlnaHQgKyBzcGFjaW5nCiAgICBjb25zdGV4cHIgaW50IG5fd2lkZ2V0cyA9IDY7CiAgICBhdXRvIGNsYW1wX3Njcm9sbCA9IFsmXShzaXplX3QgY291bnQsIGZsb2F0IHkpIHsKICAgICAgICBjb25zdCBhdXRvIG0gPSBoZWlnaHQgKiAoc3RhdGljX2Nhc3Q8aW50Pihjb3VudCkgLSBuX3dpZGdldHMgKyAzKTsKICAgICAgICBpZiAobSA8IDApIHJldHVybiAwLmY7CiAgICAgICAgcmV0dXJuIHN0ZDo6Y2xhbXAoeSwgMC5mLCBtKTsKICAgIH07CiAgICBhc3NlcnQoY2xhbXBfc2Nyb2xsKDAsIDk5OS5mKSA9PSAwLmYpOyAgIC8vIGVtcHR5IGxpc3QgcGlucyB0byAwCiAgICBhc3NlcnQoY2xhbXBfc2Nyb2xsKDMsIDk5OS5mKSA9PSAwLmYpOyAgIC8vIHNob3J0IGxpc3QgcGlucyB0byAwCiAgICBhc3NlcnQoY2xhbXBfc2Nyb2xsKDIwLCAtNTAuZikgPT0gMC5mKTsgIC8vIG5lZ2F0aXZlIHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgMWU5ZikgPT0gaGVpZ2h0ICogKDIwIC0gbl93aWRnZXRzICsgMykpOyAvLyBjYXBzIGF0IG1heAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgMTAwLmYpID09IDEwMC5mKTsgLy8gaW4tcmFuZ2UgdW50b3VjaGVkCgogICAgLy8gNi4gcmVzdWx0IGxhYmVsIHBsdXJhbGl6YXRpb24KICAgIGF1dG8gbGFiZWwgPSBbXShzaXplX3QgbikgewogICAgICAgIHN0ZDo6c3RyaW5nc3RyZWFtIHM7IHMgPDwgbiA8PCAiIHJlc3VsdCI7IGlmIChuICE9IDEpIHMgPDwgInMiOyByZXR1cm4gcy5zdHIoKTsKICAgIH07CiAgICBhc3NlcnQobGFiZWwoMCkgPT0gIjAgcmVzdWx0cyIgJiYgbGFiZWwoMSkgPT0gIjEgcmVzdWx0IiAmJiBsYWJlbCg1KSA9PSAiNSByZXN1bHRzIik7CgogICAgLy8gNy4gZXhwby1vdXQgZWFzaW5nOiByZW5kZXJzIGVhc2UgdG93YXJkIGEgdGFyZ2V0IHRoZSBzYW1lIHdheSB0aGUgbW9kCiAgICAvLyAgICBkb2VzLCBtdXN0IGJlIG1vbm90b25pYywgZnJhbWVyYXRlLWNvcnJlY3QsIGFuZCBsYW5kIGV4YWN0bHkgb24gdGFyZ2V0CiAgICBhdXRvIGVhc2VTdGVwID0gW10oZmxvYXQgY3VycmVudCwgZmxvYXQgdGFyZ2V0LCBmbG9hdCBkdCkgLT4gZmxvYXQgewogICAgICAgIGNvbnN0IGZsb2F0IGZhY3RvciA9IDEuZiAtIHN0ZDo6ZXhwKC0xOC5mICogZHQpOwogICAgICAgIHJldHVybiBjdXJyZW50ICsgKHRhcmdldCAtIGN1cnJlbnQpICogZmFjdG9yOwogICAgfTsKICAgIHsKICAgICAgICBmbG9hdCBwb3MgPSAwLmYsIHRhcmdldCA9IDMwMC5mOwogICAgICAgIGZsb2F0IHByZXYgPSBwb3M7CiAgICAgICAgaW50IHN0ZXBzID0gMDsKICAgICAgICB3aGlsZSAoc3RkOjphYnModGFyZ2V0IC0gcG9zKSA+IDAuMDFmICYmIHN0ZXBzIDwgMTAwMDAwKSB7CiAgICAgICAgICAgIHBvcyA9IGVhc2VTdGVwKHBvcywgdGFyZ2V0LCAxLmYgLyA2MC5mKTsKICAgICAgICAgICAgYXNzZXJ0KHBvcyA+PSBwcmV2IC0gMWUtNmYpOyAgIC8vIG5ldmVyIG1vdmVzIGJhY2t3YXJkcwogICAgICAgICAgICBhc3NlcnQocG9zIDw9IHRhcmdldCArIDFlLTZmKTsgLy8gbmV2ZXIgb3ZlcnNob290cyAoZXhwbyBvdXQpCiAgICAgICAgICAgIHByZXYgPSBwb3M7CiAgICAgICAgICAgICsrc3RlcHM7CiAgICAgICAgfQogICAgICAgIGFzc2VydChzdGQ6OmFicyhwb3MgLSB0YXJnZXQpIDwgMC4wMWYpOwogICAgICAgIC8vIGEgMzBmcHMgcnVuIHNob3VsZCBuZWVkIGZld2VyIHN0ZXBzIHRoYW4gdGhlIHNhbWUgZGlzdGFuY2UgYXQgNjBmcHMKICAgICAgICBmbG9hdCBwb3MzMCA9IDAuZjsKICAgICAgICBpbnQgc3RlcHMzMCA9IDA7CiAgICAgICAgd2hpbGUgKHN0ZDo6YWJzKHRhcmdldCAtIHBvczMwKSA+IDAuMDFmICYmIHN0ZXBzMzAgPCAxMDAwMDApIHsKICAgICAgICAgICAgcG9zMzAgPSBlYXNlU3RlcChwb3MzMCwgdGFyZ2V0LCAxLmYgLyAzMC5mKTsKICAgICAgICAgICAgKytzdGVwczMwOwogICAgICAgIH0KICAgICAgICBhc3NlcnQoc3RlcHMzMCA8IHN0ZXBzKTsKICAgICAgICBhc3NlcnQoc3RkOjphYnMocG9zMzAgLSB0YXJnZXQpIDwgMC4wMWYpOwogICAgICAgIHN0ZDo6Y291dCA8PCAiICBlYXNpbmc6IDYwZnBzICIgPDwgc3RlcHMgPDwgIiBzdGVwcywgMzBmcHMgIiA8PCBzdGVwczMwIDw8ICIgc3RlcHNcbiI7CiAgICB9CgogICAgLy8gOC4gbGFiZWwgZml0OiBtaXJyb3JzIFNpbXBsZVNvbmdXaWRnZXQ6OmZpdExhYmVsIC0gc2hyaW5rIGZyb20gdGhlIGJhc2UKICAgIC8vICAgIHNjYWxlIGRvd24gdG93YXJkIG1pblNjYWxlLCB0aGVuIHRydW5jYXRlIHdpdGggIi4uLiIgc28gdGV4dCBjYW4KICAgIC8vICAgIG5ldmVyIHJ1biB1bmRlciB0aGUgcm93J3MgYnV0dG9ucwogICAgewogICAgICAgIGF1dG8gd2lkdGggPSBbXShjb25zdCBzdGQ6OnN0cmluZyYgcykgeyByZXR1cm4gMTAuZiAqIHMuc2l6ZSgpOyB9OyAvLyBmYWtlIGZvbnQ6IDEwcHgvY2hhcgogICAgICAgIGF1dG8gZml0ID0gWyZdKGNvbnN0IHN0ZDo6c3RyaW5nJiB0ZXh0LCBmbG9hdCBiYXNlLCBmbG9hdCBtYXhXLCBmbG9hdCBtaW5TKSAtPiBzdGQ6OnN0cmluZyB7CiAgICAgICAgICAgIGNvbnN0IGZsb2F0IHNjYWxlID0gc3RkOjptaW4oYmFzZSwgbWF4VyAvIHdpZHRoKHRleHQpKTsKICAgICAgICAgICAgaWYgKHNjYWxlID49IG1pblMpIHJldHVybiB0ZXh0OyAvLyBzaG93biB3aG9sZSAocG9zc2libHkgc2hydW5rKQogICAgICAgICAgICBzaXplX3QgbG8gPSAwLCBoaSA9IHRleHQuc2l6ZSgpOwogICAgICAgICAgICBzdGQ6OnN0cmluZyBiZXN0ID0gIi4uLiI7CiAgICAgICAgICAgIHdoaWxlIChsbyA8PSBoaSkgewogICAgICAgICAgICAgICAgY29uc3Qgc2l6ZV90IG1pZCA9IGxvICsgKGhpIC0gbG8pIC8gMjsKICAgICAgICAgICAgICAgIHN0ZDo6c3RyaW5nIGNhbmRpZGF0ZSA9IHRleHQuc3Vic3RyKDAsIG1pZCkgKyAiLi4uIjsKICAgICAgICAgICAgICAgIGlmICh3aWR0aChjYW5kaWRhdGUpIDw9IG1heFcgLyBtaW5TKSB7IGJlc3QgPSBjYW5kaWRhdGU7IGxvID0gbWlkICsgMTsgfQogICAgICAgICAgICAgICAgZWxzZSB7IGlmIChtaWQgPT0gMCkgYnJlYWs7IGhpID0gbWlkIC0gMTsgfQogICAgICAgICAgICB9CiAgICAgICAgICAgIHJldHVybiBiZXN0OwogICAgICAgIH07CiAgICAgICAgY29uc3RleHByIGZsb2F0IGtNYXggPSAyNDAuZiwga01pbiA9IDAuNWY7CiAgICAgICAgLy8gc2hvcnQgdGV4dCB1bnRvdWNoZWQKICAgICAgICBhc3NlcnQoZml0KCJTaG9ydCIsIDAuOGYsIGtNYXgsIGtNaW4pID09ICJTaG9ydCIpOwogICAgICAgIC8vIDI4MHB4IGZpdHMgdGhlIGZ1bGwgYnVkZ2V0IGF0IGJhc2Ugc2NhbGUsIHNvIGl0IHN0YXlzIHdob2xlCiAgICAgICAgc3RkOjpzdHJpbmcgbWVkaXVtKDI4LCAneCcpOwogICAgICAgIGFzc2VydChmaXQobWVkaXVtLCAwLjhmLCBrTWF4LCBrTWluKSA9PSBtZWRpdW0pOwogICAgICAgIC8vIDQwMHB4IG5lZWRzIGEgc2hyaW5rIHRvIDAuNiwgYnV0IHN0aWxsIGZpdHMgd2hvbGUgYWJvdmUgbWluU2NhbGUKICAgICAgICBzdGQ6OnN0cmluZyBzaHJpbmsoNDAsICd5Jyk7CiAgICAgICAgYXNzZXJ0KGZpdChzaHJpbmssIDAuOGYsIGtNYXgsIGtNaW4pID09IHNocmluayk7CiAgICAgICAgLy8gNjAwcHggY2Fubm90IGZpdCBldmVuIGF0IG1pblNjYWxlIC0+IGN1dCB3aXRoIGVsbGlwc2lzCiAgICAgICAgc3RkOjpzdHJpbmcgaHVnZSg2MCwgJ3onKTsKICAgICAgICBzdGQ6OnN0cmluZyBjdXQgPSBmaXQoaHVnZSwgMC44Ziwga01heCwga01pbik7CiAgICAgICAgYXNzZXJ0KGN1dC5zaXplKCkgPCBodWdlLnNpemUoKSk7CiAgICAgICAgYXNzZXJ0KGN1dC5zaXplKCkgPj0gMyAmJiBjdXQuc3Vic3RyKGN1dC5zaXplKCkgLSAzKSA9PSAiLi4uIik7CiAgICAgICAgYXNzZXJ0KHdpZHRoKGN1dCkgPD0ga01heCAvIGtNaW4gKyAxZS0zZik7IC8vIGZpdHMgdGhlIGJ1ZGdldCBhdCBtaW5TY2FsZQogICAgICAgIHN0ZDo6Y291dCA8PCAiICBsYWJlbCBmaXQ6IDYwLWNoYXIgbmFtZSAtPiAiIDw8IGN1dC5zaXplKCkgPDwgIiBjaGFycyB3aXRoIGVsbGlwc2lzXG4iOwogICAgfQoKICAgIHN0ZDo6Y291dCA8PCAiQUxMIExPR0lDIFRFU1RTIFBBU1NFRCIgPDwgc3RkOjplbmRsOwogICAgcmV0dXJuIDA7Cn0K",
 "about.md": "PiBUaGlzIG1vZCB3YXMgcG9ydGVkIHRvIGdlb2RlIHVzaW5nIEFJLiBJZiB5b3UgZG9uJ3QgbGlrZSB1c2luZyBBSSBwb3J0ZWQgbW9kcywgZmVlbCBmcmVlIHRvIG5vdCB1c2UgaXQuCgojIFNvbmcgU2VhcmNoCgpBbiBpbi1nYW1lIHNlYXJjaCBkaWFsb2cgZm9yIHlvdXIgZG93bmxvYWRlZCBzb25ncywgd2l0aCBmdXp6eSBtYXRjaGluZyBhbmQKSnVrZWJveCBOT05HIHN1cHBvcnQuCgpUaGUgc2VhcmNoIGRpYWxvZyByZXBsYWNlcyB0aGUgZGVmYXVsdCBzb25nIGJyb3dzZXIgaW4gdGhlIGVkaXRvcidzIHNvbmcgbWVudS4gQXMgeW91CnR5cGUsIHNvbmdzIGFyZSByYW5rZWQgYnkgaG93IHdlbGwgdGhleSBtYXRjaCDigJQgbWF0Y2hpbmcgYWNyb3NzIGJvdGggdGhlIHNvbmcgbmFtZSBhbmQKdGhlIGFydGlzdCDigJQgc28gc2hvcnQsIGFwcHJveGltYXRlIHF1ZXJpZXMgc3RpbGwgZmluZCB0aGUgcmlnaHQgdHJhY2suCgojIyBGZWF0dXJlcwoKLSAqKkZ1enp5IHNlYXJjaCoqIGFjcm9zcyBzb25nIG5hbWUgYW5kIGFydGlzdAotICoqTG9uZyB0aXRsZXMgdGFtZWQqKjogbmFtZXMgc2hyaW5rIGR5bmFtaWNhbGx5IHRvIGZpdCB0aGVpciByb3csIHNvIHRoZSBmdWxsCiAgdGl0bGUgc3RheXMgdmlzaWJsZTsgb25seSBhYnN1cmRseSBsb25nIG5hbWVzIGZhbGwgYmFjayB0byBhIGNsZWFuICIuLi4iIGN1dAotICoqU29uZyBwcmV2aWV3Kiogd2l0aG91dCBsZWF2aW5nIHRoZSBkaWFsb2csIHdpdGggdGhlIHBsYXkgYnV0dG9uIHVwZGF0aW5nIHBlciByb3cKLSAqKkRlbGV0ZSBzb25ncyoqIHN0cmFpZ2h0IGZyb20gYSByZXN1bHQgcm93ICh3aXRoIGEgY29uZmlybWF0aW9uKQotICoqSnVrZWJveCBpbnRlZ3JhdGlvbioqOiBlYWNoIHJvdyBjYXJyaWVzIHRoZSBzYW1lIHBpbiBkaXNjIEp1a2Vib3ggZHJhd3Mgb24gaXRzCiAgb3duIHNvbmcgd2lkZ2V0cywgcGlubmVkIG9uIHRoZSByb3cncyB0b3AtbGVmdCBjb3JuZXI7IGNsaWNraW5nIGl0IG9wZW5zIEp1a2Vib3gncwogIG5vbmcgcGlja2VyIGZvciB0aGF0IHNvbmcuIFRoaXMgaXMgZW50aXJlbHkgb3B0aW9uYWwg4oCUIHdpdGhvdXQgSnVrZWJveCBpbnN0YWxsZWQKICBub3RoaW5nIGNoYW5nZXMgYW5kIHRoZSBtb2Qgd29ya3MgdGhlIHNhbWUuCi0gKipTZXR0aW5ncyoqIGZvciBldmVyeSBwYXJ0IG9mIHRoZSBhYm92ZSwgaW5jbHVkaW5nIGZ1bGx5IGRpc2FibGluZyB0aGUgbW9kCgojIyBDcmVkaXRzCgotICoqbWF0Y29vbCoqIOKAlCB0aGUgb3JpZ2luYWwgbW9kLCBpbiBbc21hbGwtZ2QtbW9kc10oaHR0cHM6Ly9naXRodWIuY29tL21hdGNvb2wvc21hbGwtZ2QtbW9kcykKLSAqKkZsZXltIC8gRmxhZnkqKiDigJQgW0p1a2Vib3hdKGh0dHBzOi8vZ2l0aHViLmNvbS9GbGVleW0vanVrZWJveCkgYW5kIEF1dG9Ob25nLCBmb3IgdGhlCiAgTk9ORyBzdXBwb3J0CgojIyBQb3J0aW5nIG5vdGVzCgpUaGlzIGlzIGEgcG9ydCBvZiB0aGUgb3JpZ2luYWwgV2luZG93cy1vbmx5IE1pbkhvb2stYmFzZWQgRExMIHRvIGEgcHJvcGVyIEdlb2RlIG1vZC4KVGhlIG9yaWdpbmFsIGhvb2tlZCBgQ3VzdG9tU29uZ0xheWVyOjpvblNvbmdCcm93c2VyYCBhdCBhIGhhcmRjb2RlZCBvZmZzZXQ7IHRoZSBwb3J0CnVzZXMgR2VvZGUncyBgJG1vZGlmeWAgaG9vayBpbnN0ZWFkLCB3aGljaCBpcyB3aGF0IG1ha2VzIGl0IHdvcmsgYWNyb3NzIEdEIHZlcnNpb25zCmFuZCBwbGF0Zm9ybXMuCgpUaGUgTk9ORyBpbnRlZ3JhdGlvbiBkZXNlcnZlcyBhIG5vdGU6IEp1a2Vib3ggZG9lcyBub3QgZXhwb3NlIGEgcXVlcnkgQVBJIGZvciBvdGhlcgptb2RzLCBhbmQgKHZlcmlmaWVkIGFnYWluc3QgaXRzIDMuOC4wIHNvdXJjZSkgaXRzIHBpbiBkaXNjIGlzIGRyYXduIG9uICpldmVyeSogc29uZwp3aWRnZXQgdW5jb25kaXRpb25hbGx5IOKAlCBpdCBpcyB0aGUgYnV0dG9uIHRoYXQgb3BlbnMgaXRzIG5vbmcgcGlja2VyLCBub3QgYSAidGhpcyBzb25nCmhhcyBhIG5vbmciIGJhZGdlLiBTbyB0aGUgcG9ydCBtaXJyb3JzIHRoYXQgbG9vazogaXQgYnVuZGxlcyBhIGNvcHkgb2YgdGhlIGRpc2MgYXJ0d29yawooTUlUKSwgcGlucyBpdCBvbiBlYWNoIHJvdydzIHRvcC1sZWZ0IGNvcm5lciwgYW5kIOKAlCB0byBtYWtlIGl0IGNsaWNrYWJsZSDigJQgYnVpbGRzIGEKdGhyb3dhd2F5IGBDdXN0b21Tb25nV2lkZ2V0YCBmb3IgdGhlIHJvdydzIHNvbmcgYW5kIGFjdGl2YXRlcyB0aGUgZGlzYyBidXR0b24gSnVrZWJveAphdHRhY2hlcyB0byBpdC4gVGhlIHBpY2tlciBrZWVwcyB0aGF0IHdpZGdldCBhbGl2ZSB3aGlsZSBpdCBpcyBvcGVuLCBzbyB0aGVyZSBhcmUgbm8KaG9va3MgaW50byBKdWtlYm94IGFuZCBubyBzdG9yYWdlIGd1ZXNzaW5nLgoKQ29udHJpYnV0aW9ucyBhbmQgYnVnIHJlcG9ydHMgYXJlIHdlbGNvbWUg4oCUIGlmIHNvbWV0aGluZyBsb29rcyB3cm9uZywgaXQncyBsaWtlbHkgaW4KdGhpcyBwb3J0IHJhdGhlciB0aGFuIHRoZSBvcmlnaW5hbC4K",
 "resources/nong_disc.png": "iVBORw0KGgoAAAANSUhEUgAAAJYAAACXCAYAAAD3XaJHAAAACXBIWXMAABFtAAARbQFSEUH3AAAAGXRFWHRTb2Z0d2FyZQB3d3cuaW5rc2NhcGUub3Jnm+48GgAAIABJREFUeJztfXl8VNXZ//feO/uafQOysiRhURQkRNktYD9oKxZ9q6JWUd/+XH4uL4i20bb201pblLq0tgiCb9v3hxB8hVpA2WtEQBazsEMSSCCBTCbJTDL7Pb8/ciczyT33ZibbTCDfz+d8IOfeM3Pm3O99znOe85znYQghuA4xAsBIAJlBJb5LYQBoAGiFNnYAHgBeAJagcgVAVVA5KdRf12CuA2JlASgEcAuACQBuABDbz99ZC6AMwHcA9gP4GsDVfv7OqMK1SKwUAPOFMh1AaqQ7JOAUgL0AtgHYCaAl0h3qT1wrxBoF4D4ACwHcKExj0QwPgK8AbARQDKA+0h3qawxmYiUAeAjAgwAmRrozvYAPwB4AHwPYAMAR6Q71BQYjsWYCeBLA3QDUke5MH6MJwN8A/AVAeaQ70xsMFmIpAfwQwIsApkS6MwOEEgC/A/BPAIPiIQUj2omlAvAYgJcFE0GfobKyEmfOnEFVVRWqq6tx8eJFWCyWjuL1etHc3NxxP8MwiImJgUKhQHx8PBISEhAfH4+MjAxkZmYiMzMTubm5SElJ6ctuAsBRAL8AsGUwESxaicUBeBhAkWBj6hWqq6tRUlKC/fv34+jRoygrK0NLS/8sypKSkjBhwgTcdNNNKCwsxNSpU5GUlNQXH31IGI/tffFh/Y1oJNYsAG8L9qYeoampCTt27MC2bduwfft21NTU9G0Pw0Rubi7mz5+P+fPnY8aMGdBoNL35uJ0Anot6HYwQEi0lnRCyifQQVquVrFu3jixYsICoVCoiTBtRV7RaLVmwYAFZt24dsdvtPf25bkLISkKIMQqeG7VEvAOEEIYQ8gQhpCXc0eV5nuzYsYPcd999RK1WR5w04Raz2Ux++tOfkiNHjvSUYJcIIXdHwTMUlUhPhaMArAMwNZxGdrsdH330Ed555x2cPXu2Vx3gOA5GoxEajQYcx8Hn88Hj8YAQAofDAYZpt7UyDAO1Wg2WZaFUKsEwDHieR1tbG2w2G3o7jpMnT8YLL7yAH/3oR1AoFOE2/x8ATwGw9qoTfYkIsnoJIcQWzuvZ0NBAXn75ZRIbG9sjCWEwGEhqamqP23dXOI4jCQkJJDU1lWg0mh59RkZGBlm5ciVpa2sLV3pdIITMirSk8pdIfKmZEFIczohZrVZSVFREjEZj2PpMampqRKdJvV5PUlNTiVKpDKtdWloaef/994nL5QpnqHyEkN8QQrjrjVhjCSGnQh0lj8dD3n33XRIfHx/Wg0xKSiIMw0SMTHISLTU1NazFRU5ODtm0Kew1zQ5CSOL1QqxF4Ux9O3fuJPn5+SENPsMwJCUlhXAcF3HyhFo0Gg1JSkoK+f7Zs2eTioqKcMhVTQi5+Von1v8VxHS3aGxsJI8++mhIEker1YYlzaK1JCUlEYVC0e19arWa/PKXvwxnerQTQn5wLRJLQQhZFeoofPHFFyQ1NbXbAdbr9WHrW4OhxMfHhzRNjhs3jpSXl4c6rF5CyLPXErHUhJD/DeWXezwe8rOf/YywLCs7oFqtlphMpogToL9LQkJCtxJMp9ORNWvWhEouQgh5/Voglo4Qsi2UX1tXV0duv/122UFkWZbExcVF/IEPdElLS+v2nsWLFxObLWTV9W3BID0oiaUjhOwO5Vdu3bqVJCYmyg5cfHx8VK7wBqqoVCpiNptl78nNzSWlpaWhkutPA0Guvv5AdaiS6q233pIljFKpvC6mvVBLdy+YTqcjW7ZsCZVcKwYTsZSh6FQ8z5PXXntNdhCNRuN1LaWkikqlIjqdTvI6x3Fk1aqQ10q/HCzE6vYXeb1e8vjjj8sO3pCU6r7IbUkxDENee+21kJjl8/n+M9qJtby7H+F0Osk999wjOSBKpVL2bRwqYnLJraKfffZZ4vPJmw55nvd6vd47opVYiwghvNwPaGtrI9OnT5cchJiYGHLXXXeRKVOmRPyBDaaiVqtl9yAfffRRwvOyj4Z4vd5Wu91+Q18Ti+2lc8RYAGvkzvH5fD48+OCD2LdvH/V6QkIC5syZg/j4eOTn52P06NG97NL1A5fLBZ/PB61WS72+Zs0aLF++XPYzOI7TsSz7z+bm5j49Hd4bYhmFc3AGqRsIIXjyySexadMm6vXU1FTMnj0bJpOpo66goACJiYm96Nb1BZ7n4XA4YDQaqdfffPNNrFixQvYztFrtcI/H89mGDRu4vupXb4i1FkCe3A3Lly/H6tWrqdcyMzMxa9Ys6PX6TvUcx2H27NnQ6XS96Nr1B5vNhthYutBZunQp1q5dK9s+ISFh2tixY+UZGAZ6SqwlwnF2Sbz99tt48803qddycnIwffp0qNX086Y6nQ6zZs0Cy/Z2pr6+YLVaYTabRfWEEDzxxBPYtm2bbPu8vLynN27cKPtcQ0VPXJNzhLNudNkLYNeuXfje974HnudF19LS0jBr1iyoVKpuv+jkyZPYv39/uP277mEwGGC326n1hw4dQm5urmRbu91e9/TTT9+2du3ac73pQ7jEYoVgFpI+6nV1dZg4cSLq6upE1xITE8Oe5kpKSnD69Olw+jgEQeq3tbWJ6sePH48DBw5IKvwAUFZW9q8JEybcSwhp7en3hzvXPC1HKp7n8cADD1BJZTKZMG3atLB1p6lTpyI5OTnMbg7B6XRCqVSK6svKyvDiiy/Kth03btwdv/vd7x5nGCbsUx1+hEOsdAC/lrvh1Vdfxa5du0T1KpUKM2fOpM7/3XaQZTFr1qwhZT5M8Dwvedrnz3/+M9avXy/ZlmEY5sknn3wxIyNjck+/Pxxi/VFOr9q9ezfeeOMN6rWCggLEx8f3pH9A+3IYs2fPBsf12Wr4uoDD4ZBcKS5ZsgSnTp2SbGs2m4evWbPmaYZhhvfku0Ml1mwh2gsVVqsVDzzwAHw+n+haXl4ecnJyetK3TkhMTERhYWGvP+d6g9VqRUxMjKjebrfj0UcfhZyOPX369IVz586dwzBM2NNFKMTihFgKkli6dCkuX74sqk9KSsJNN90Ubp8kMXLkSNkVzRDoaGpqoirrX3/9tax9S6FQaFauXPkIgLAfYijEelgICkvF7t27sWbNGlG9UqlEQUFBSGaFcDBlypT+CBV0zUNKMr300ktobGyUbJebmzvjpz/96S0Mw2SE833dEUsJ4OdSF71eL5555hlqp6dMmdIrvUoKLMti5syZIov9EOThdDqpW2VXr17FK6+8ItmOYRhm2bJl9wEYzzBMyBEUuyPWEiGcNRUffPABKioqRPUjRozoE71KClqtFnPmzBlS5sPE1atXqbsdq1atwjfffCPZLjMz85Zly5aNBTA+1O+SM5AqAZyTiqTX3NyMnJwcWCydY+WzLIsFCxb0i7TqinPnzkl6TQyBDo1GA6fTKaqfPHkyDhw40BEEpSsuXLhwOCMj4zcA9hBCpOdOAXIS6z/kwjO+9dZbIlIBwMSJEweEVBD2HPPyZPfBh9AFTqcTcXFxovpDhw7h008/lWyXnp5+02OPPTZCTt8OhpzEOiIV5tpisSA7O1sUblGn0+HOO+8cUGMmz/P44osvqKvSIdDBsmzA0zMIY8eORWlpqeTmf0VFxc5x48b9CcBBQohsmEQpiTVTLnb622+/TY3heeONNw64hdxvmZfyRxqCGDzPU+OiVlRUoLi4WLJdXl7e9MLCQjOAfEZqzhQgRaz/lGrQ1taGDz74QFRvMpmQnZ0t9139BrVajdmzZ/ckYNl1i/r6eqpkknJ1QvtLrHz11VdnCs6dslGsacRKkLOyf/TRR1Tdavz48dRNz4FCXFzckGU+TNCk1rfffiu7IJo6dertHMcxAHLlpBaNWA9JZXwghOCdd94R1ev1emRm9jpqdq+Rk5ODsWPHRrobgwY0LxQAWLlypWQbk8mUVlRUlCdILckEWDRiLZa6ec+ePVTfqHHjxvW5hb2nmDRpElJToyXhV/SDtouxZcsW2cXQwoULpwv/HSV1T1dijRKyZ1GxatUq8QewLNLT0yU7MdDoa2Veo2Ch4q5dF+mrV8VpFL1eL9atWyfZZsyYMVPMZjMHIJ5hGLHtgmJuKALwK9qNzc3NSE5Ohsvl6lSfn5+PKVOiL72N1WrF5s2bqe7RNKQa1BibYMCYeD2GGzVI1qtgVAUWA4QAzS4PLtldqLE5caKhFeUNdjQ5Pf34KwYGMTExaGpq6lQ3ZswYnDx5UrLNH/7wh9eXLl16DEA1IeRw1+tdl1F3S33Qp59+KiIVBB/2aAMhBMePH++WVCa1AjPT4zAjPQ7pJvlsEQwDxGiUiNEokZ9gwNysBBAAJxrs2HOhESU1TXD7QiNxtIGmg586dQrHjh3DjTfSJ7D58+cXCMQazjBMKSGk0xsWTKwUuWlww4YNojqTyYRhw4aF+TP6F16vF3v27MHFixcl7zGrFbh7dDJuz4qHuhfTHAMgP8GA/AQDHhibhs1nrmDr+QZ4BhnBrFYrGIYRGUzXr18vSazs7Gy/Kw0nmB7OB18PngofAfAR7UNaWlqQmJgIt9vdqf7mm2/GhAkhWfgHBDzPY+fOnZK5cxgGmJuVgB/np0KvpG9gq9UEGjWBUkWgULS3ISAgPOD1snB7AKeDgcdDX2nXt7qw+rtaHK0fXJl5k5KScOXKlU51o0aNkj3IsmTJkudWr159EYCFELI3+FqwxJov9QE7duwQkQrC8fhoQklJiSSpjCoFnpmUjonJJtE1nZbAaOKh1/OQP8oY8JD1+gC7jUVLCwu3O0CyZL0arxRmY0eVBWu+q4GHD/t4XURA25g+c+YMzp49i5EjR1LbLFq06CaBWPEMw2gJIR3ZYYOH8TapL6UddNTpdFHlcFdRUSGZ/iTdpMHvZ48RkUpv4DFihBdpw7wwGrsjVWcoOCAmhkd6uhdpaT605ygI4PbMePxi2shOC4BohlSave3bpbPY5ebm5gf92ck33j8VZnWdI4ORmZmJ6urqTnUTJkzAzTffHHrP+xENDQ34/PPPqcr66Dg9XinM7jT1KRQESUk8dDrx/VftDM5aGFQ1s6ht5WD3Mu2Ku4pghMGHLDOPnDgepi6evgSArZlFg4VDcDdqbU68XnIOFkf0rx7j4uJE3qR33nknNm/eTL3f7XbbdDrdT3w+H+k6HfpfJ8m9kJqaGhGpAEie/hho+Hw+7N27l0qqdJNGRCq9niAp2Ytgnd3pAfZUc/iiWo1WrxoKTgGO48CyXPuKiQBWB49zrTx2XPLC6/UiRePC97PdmDTcBwXbrsibzDw0OoL6OgX8C+hhRg1+fmsOivadgd0tPmwSTaBtyZWUlIAQQl05qlQq44MPPjhs3bp1NQDiGIZREULcCJoKb5H6sq+++opaHy36lVS2VLNagVcKczqRymzikZoaIJXPB3x+WoHndxmxpToORBkHk8EMnVYPtUoDpUIJBaeAglNAqVBBrdJArzXAbIxBCxOHj0/H4oWdenxbE2CpSkkwbJgHWm1gahxu1GBZQTY4eYeAiKOhoUFU19jYiBMnTki2mTdvnl8BYwB0bD76R0QymynNZdVkMnUKPRQptLW1obS0VFTPMMDTN2cgXht4A80mHolJAYnRYGfws306fH4hBjpdDHQaPRRc6PqQUqGEXmsAp47F6uMxWPGNCg5hfcOyQFqatxO58uL1+I/86NFJaaAd34MgtaQwcuTI4E3ijiPrfmJJ+jIfPXpUVBctWzjl5eXUwZiblYAbkwNbOno96USqsw0MXi0xoA1m6LUGsGzPfecVnAIGvQlVbbH42b/1sLb68xsCqaleqJQBcv1gVDJGx0X3IRCainPs2DHJ+9PS0oJP73RMY6ygzVP3ewghVIkQDdLK6XRST/Ka1Qr8OD+wCa1QtOtUfpxtYPDmtyaoNSaoVSEfOukWGrUWXtaE177Wo0mIxcGyQEqKD+0BoNvJ9viNw8FG8ZRIc9QsKyuTvD8hISFYyugZhtFCIJbkDnVNTY1oDwmCQ36kcfbsWXi9XlH93aOTO+lVSUl8h07VYGfwhyMG6LUGKBR97zumVqkBhRG/3q+DW+iaSk0QGxdYWGSatbhtuPhkcrSAZs8qKyuTPJeoVqtjxowZE7xGjoNALElHKimrazSsCM+dE4dvMqkVuD0rcJBDbwiYFHw+YMW3WqiU/UMqP9QqNZwwYNXRwHfExvBQBk2Jd49JRrQKLdrh1aamJqoXhB+zZs0K9hjsIJbkucHKykpRHcuyMBgkw44OCJqamqgDMDM9rtPeX1xsQFJsO6dAs1fXp9OfFLQaHcqsBpy6EtC3YoP6MtyowbiE6PTRl5JMVVVVkm3GjRsXHGfKDIFYkpo4zX4VHx8f8RCOtbW11PoZ6QFVUaclHdZwhwf4Z6UWWrV0sLG+hkatxUfHtfA/J6ORR/D52unpkZf64UCOWBkZGcFHrE0QiCUZopi27xYN0yDNpTbVoO7k+mI0BSTE3moOrELTq9VfuFAqVGhya3DGEpBaRmOgT5NTzVGrxNMUeDlvkZiYmODVnIZhGDUbvETsCprBLBoCoHXdhQeAsQmdp2e9PvAQv6hWQ6Xs/ymwK1RKNb6oDNjGgvukV3LIihk4CRoOaM+YdoDGD6PYXdcQNrEi7dvucrmoK5cx8QH7kFpNOjaUr9qZjm2agYZKqUJ5oxZ+9yyNhoAJ0iLGRKlNi7a1Q+OCHwaDoSux9KxclD6r1Sqqi/TZvebmZmr9cGNgGtQEeRqctTARIRX8npmMAvUtgekw2GA6zDjwUjQU0HRouVBHarW6q4jTsVJHvSCEGuyKbg7A9jtokYABIEUf+BlKVeDhVbawEY1Kw7EcLtkDYxYs8FMN0UEshmGgUqmg0WhgNBqpwoPGBT84jusq4nQKAJJzG825L9IrQo9H7H7CCDqLH8ECqtbODajS3hUsy+JKW4BYHEc6Ug/plXRJyrIsFAoFFIp2Lwt/6cu/VSpVp7+DUV5eLrII0LgQ+E2iKUEdNrEiDZq1XcWxnQyObND/23xMRKUswzCwB7kxB7+XWgX9JZ03b15EnShpwoN2kMYPhdjirGIFHzUqIj3t0UAz4Ilqgiqi4USgotMwBjrHSxgjo3Hc5frE8yL/a5UCgBsAdd1LWwEGP1ifzwefzwev19vxf6k6/99ut1vymtzfXq9X8jiX28fDR0iHv1PwAzMoCSzOyJ2aIYRAH6Sw83zgATm89H5FMgYGJNxnpPIeCfd31U+UssSifVhFRYWk818kYXN5EaNpfyBeL9tx8CHd4MX5Nh6RelQ+3ocUfYBYwTO5zS2e1hEFxKLNCnLE4nm+6w/hWABio5AAmqEsWuN+XrIHdAB30PuTHUPg89Ef4EDA5/NhmDHwoIJP9FyyifUWhmEiboSm6dZyuXe8Xm/XBiwLQOwXI4AW8jHSdiwp1NgC74fTEXh4OXE8VeEfCPA8DwXjQbyBCH8zcAcp8sF99sNgMET85aWZFuSSkzocjq7JnFgWgKRJlUasaFQsAeCEJfDbPB4GXkFNMGmBFI0LHu/An5Jxe9woSHZ3rFgdjs4Li4oGceo3WhaJgQYtJZ1cXFm73S46dMACkNwEorE01CAbA43yq/ZOq0O7PbAe/H62G26P9HK5v+D2uDAnM0Bouz0giSwODy7bxX2KhkxntN0NucMzdjETeRaAeEdXwPDh4vw8UpbvSKPJ6cGJIAnQ0hwg1qThPiiIA94B1LVcbheyjG1IM/unQaC1NSDt99fSNZBIx/ZyOp3UqVDunENDQ0NXJvIsALHTlYCsLLEPIE1MRgv2XgjsbbrdDNoEi7eCBRbnOeB0DsxLQQiBy+XAI+MCOm1TE9vpIOveC+K9N41GM2ChzKVgs9mo9XIRGysrK7sKJx8LQOwmKvNhPUj1O2D4qsaKZldAKjU2ch3T4+ThPHKMrXC4pPe8+gptzlbckWlHikmQVj4GzU2BabCiwY6qZnE/srOzI67DtrbSk6rKEau0tLQrsTwsAEnXQKng/JFetUjB7eOx5UzgNzqdDFpaAlPiU5NcUBE7XO7+07cczjak62y4a0yA4A0WFsGRjYpP1lPbSgXfGEjQTA2pqalyiwqyffv2rsRyswAkw7YlJCRQ5/xoWLlIYev5BtS3BgbH0sB1LPG1SuDnBW0gHlu/kMvhbEOswoYXbnF17Fe2tnYm9+G6FpRdFU83SUlJEZ8GIaG4jx8vnULHbrdfra+v77rkdvtXhZekGtLiX0XaMiwHt4/HmtKASzXPA3WXuY6tlDg9wa9us0PFN8PRRzoXIQStDjuGa5tRdKsD/gAzbg+DK/UBu5/Lx2PNd/QwS1IBzgYaNHd0OWI1NDTQdHSX/1X6TqohLZElzQEwmnCkrgXbzgfMc243g8uXORDSTq5YHfCbGW3INTfB1toMj8hwHDpcbhds9mbMHWbFsqmuDlJ5vQwuX1J0mgJXf1eDK23i70pKSoqKyIgtLS3Uc6RyyUxrampoxGrzE0t83FkALSi/nAtFtODjslqcsQYkksPBoLY2sDJTKYCnJ7nx/MQmaEgTbK0tcLldIS1OeJ6H0+VEi70ZqepGvF7YjB/m+TqmP4+HQW2tAsGuY7uqG7G7WrwSZBgGt9wiGZNlQCHlfnzbbZKh03D69GkasVr9cnq/VMNbb70VLMuKDKN6vV5yBREN8PAEv/36PH49YxTSBE9Np5NFTQ2LlBQvVIKX6ZhEgt/OaMPpqw58UaVEhVUDMApwLAeWZTtWaYQQ+Ph2bwsF40FBshuzMz0YZu5MxNbW9ukvWFIdrW/BX4/RT7nk5ubKbpcMJGimpGHDhsnasDZu3Eg71dxBrBJhs0G01o2NjUV+fj7Ky8s71RuNxqgmFgTvgV99dRZFt+ZgmOAT73YDFy8qEBfnQ0wMD4Zp90Ufk0QwJskNH+9GXQuDS3YGV1oZOHwMAAKDAkgxEAwz8kgwQHSSmfcxaLCwnRR1CKRacaAKPkrIyJiYGEyaNKl/ByFEEEKop8tnzJgh2aa1tfXq1q1baWLO7idWA4AzAEbTPmD+/PkiYsmd2ogmWBweFO07i2UFWcgVTvIQAlgsHFpaWMTG+mA0kY43imOBYTEEw2JCs9fxPNDUzLYbQH2d2baruhF/PXaRSiqFQoFZs2ZFzaa+xWKh6lfz50uGpkVtbS0tcJaTEOIOfr32SH3AvHnzRHVerzdq7VldYXN78Yt/n8VnZ6502k/0eBhcuaJAdZUSDVc5OJ0sQrH/+rdn6us5VFUp0WjhOpHK5ePxpyMX8OcjF6ikYhgG06dPjyqzDe3cIMMwmDt3rmSb0tLS45TqZnSJmrwNwBO0D5g2bRpMJpMocl5iYqJkop9og48Q/K38Eg5dasaSG4cj0xzwL/J6BanTDDAsB7WKQKlsD4HUHoKIAc8TeL3tbi9uNyPp0H24rgUfldZ0sqV1xa233oqMjLCSwvc7aHE6Jk2aJLspvnbtWnHwNKAFXYi1E4BHyAXdCWq1GnfeeSf+/ve/d6qXO2sWrTjV2Irlu0/jthGxuHt0Uofu5Qfh2y327WdiQ99eOd5gx8aT9VTjpx8Mw6CwsBCjRklGjooILBYLNSnTokWLJNs0NTVd+Pzzz2n6UCO6EKsFwFcAZtE+6N577xURy+12Syavjmb4CMHeC43Yd7ER4xONmD4iFpNTzdBJJBWQQqPDg/21Tdh7sRGVTfJ7kBzHYdasWRgxQjZ/ZERQXy/eYmIYRpZYp06dokkr+N2wumqOG6WINW/ePMTGxoqMo0ajcdARyw9CgNIrNpResYFlGGTHaDE6rj1JU6pBDZ2ShV6pAE8IHF4fml1eXLa7cLHFiROWVtRSPECloFAoqEm+Iw2fz4eKigpRfUFBgezG84YNG76mVNsJIe2yvotBMBlArZAfRYRnn30W7777bk/6PwTBC/P73/9+1KwEIYSq2rVrl6h+1apVWLJkCbWN3W6/EhMT83+E+O7BqCKEHAHl2F293Orw8ccfp9bTUsAOQQyLxYKvv6a96JEDzXZlMBhw3333SbYpLy8voZAKAn8AifOcH0t94Pjx4zF16lRRvVyImyF0xrlz53D8OG2VPvC4fPkyNbjej3/8Y9lEoh9++OFeSjUJ9kamEWuD3Mmd559/XlTn8/miwuVjsODQoUOyqXEHCufPi7PcMAyD5557TrLNlStXTgmJmbqiMThnIY1YDgB/p9QD7fmAkZOTI6qXSvIzBDF4nsfu3bsl3YAHAnV1ddTgxQsWLEB+fj61DQDs2rXrS4lLneJ3SoU2+EAqpgPHcVSp5fF4hqRWGHC5XNi1a1fEzjxKRcReunSpZBu3221/+eWXaUoiCZVY5QAk84k9/vjj1KVotPtpRRsaGxtl04n0F6qqqqhK+/e+9z1MmzZNst3Bgwe3VVVV0XymLMG5CtFNMJYVUhdUKhWWL18uqud5PmqSNw0WnD9/XrTB35/weDySKUx+9Stqnnmg/dl6ioqKxIkr23Gha4UcsXYAkLKu4tFHH8Xo0WJniIaGhojHKR1s+PbbbyUzw/Y1jh8/Tp1Z7r77bhQUFMi127tnzx7alOQFIOp8d+Gjfil1QalUYsUKulCTi0wyBDEIIdi3b1+/K/NWqxXffSf2QtfpdHjrrbck2/E873v11VeLJS7XEEJEimJ3xNoM4JDUxQULFlDdKmw225AiHyZcLhd27tzZb8q8P+EWLfZVUVGR7PZNeXn5jk8//VTqxDz1XGp3xCIAiuRuePfdd6lJmywWS1Sf5olGWK1W7Nu3r18++9SpU1S7VV5eHl544QXJdj6fz71s2TIpadVACKGu2EKJpLgdwFapi6NHj0ZREZ17g8URMJpQXV1NTeXXG9TV1eHgwYPUa++9956sTrx///7Ptm/fLrW1ckaqXaghOl8UfLWoWLp0KfVcnNPphNlsDvErhuDHkSNHZFOMhAOHw4H9+/dTp8D7778fs2fPlmzb1tbWuHjx4k8lLtsASHp5hkqXYXZwAAALWklEQVSsEwD+JHVRqVTik08+oe4vNTc3D5ErTPiV+d7uZhBCcOTIEaove3p6ereeKps2bfpvCbsVAJwgMmflwgkqXARA8jUaNWoU3n//feq1lpaWqEieOZjgdruxY8cOalz7UHHy5EmqhV2lUmHjxo2y/mG1tbWljzzyyL8lLrd0tbR3RTjEsgF4Uu6GxYsX4yc/+YmonhAyqA5fRAuam5t7rMxXVlZSE8UDwJtvvonJkydLtvX5fK7nnnvuLxKuMQBwXE5ageLoFwr+DuB+qYtOpxMFBQVUe4lKpYrKpATRjokTJ4YV2+HChQvYvXs3NfriggULsHnzZtlwSdu2bVt3xx13bJa4bCGE0NxmOqEnxIrheb6MZVlxuD8Bx48fx5QpU6gna6P9BHU0gmEYzJkzJyR/+bq6OuzcuZP6AmdlZeHw4cOyOSfr6uqOZ2dnv+ZwOGgxQQmA3YQQSbcqP3qSuKGJZdmHAEgGI83Pz8eGDRuodqzW1tYhZT5MEEKwd+9eqhIeDIvFgn379lFJFR8fj3/961+ypHK73fZHHnlkpQSpAKA6FFKhFxlBdrtcLslNaggnaNetW0fNyzK0UgwfHo8Hu3btklQl6uvrsXPnTupsoNPpsHnzZuTm5sp9BVmzZs2fZGxWLsHrJST0ZCr0g2tra9ur0+lulbvp/fffx9NPP029ptVqZdOVDUGMESNGYM6cOZ10pJqaGuzdu5dKOqVSic8++wx33HGH7OceOHDgs4KCAkm3dACHCCEhG9d6k8PIp9Pp7nI4HLI+tk899RReeukl6jWHwwG1Wh1Vp1aiHRcvXuzk9nL+/HlJnYphGPzlL3/pllSXL18uv/322yW9hgHUh0MqQJi/e1OqqqoKvV6vg8iA53ny5JNPEkH5ExWFQkG0Wq3k9aEiLjNnziRTpkyRvM4wDPnjH/8o91gIIYTYbLb6goKCRwAslCh3AtCGy4teE4sQgkOHDj3E87yvux/xxhtvyA5WTExMxB/YYCmJiYmS1xQKBfnwww+7JZXH42ldsmTJczKkWghgWE840SfEIoTg4MGDv+/2lxBC3nvvPcKyrOSgmEymiD+0aC4syxKDwSB5XafTkc8//7zb5+Dz+Tyvv/56UTekmthTPvQZsQBwhw8f/lso5Nq0aRPRaDSSg6NSqYharY74Q4y2YjabZV/KuLg48tVXX3U7/jzP+1avXr2iG1LNAcBFnFgCufRHjhz5LBRyffnll8RsNssOZEJCQsQfZrSU7sYiKyuLnDhxIhRS8f/4xz/e6YZUCwDoesWFviQWIQR6vT7lu+++2xoKuaqqqkhBQYHsgF3v0stkMslKKQDkrrvuIhaLJZQh54uLi//SDal+CCCp10Kmr4lFCIFGo8k6ePBgSJLL7XaTl156iTDtEc4kS0xMTLcDfC0VtVpN9Hq97D0KhYK88cYbhOf57hnF877169e/1w2pFgLI6pPZqz+IJUyL4/ft2/dJKOQihJDi4uKQVoVJSUkRf+j9WTiOk13x+UtWVhY5ePBgSGPr8/k8f/3rX38fAqnG9tnz70diMQAmFRcXf8DzvDeUATh//jyZPn16t4PKsuw1RzCO40L+TQ888ACxWq0hkcrtdtt+/etfvxoCqW7q0+ffX8QKIteUN95445cej6c1lIHgeZ58/PHHJDk5OaRBTklJIRzHRZwYPS1qtZrExcWFdG9eXh7ZuXNnSIQigvHz4YcffjYEUt3i394bFMQSyMUCKHziiSeeb2lpuRzqoFitVvLUU0+FTBqTyTSoDKxJSUlEp9OFdK9erye//e1vicvlCplUly5dKi8sLPxJCKSaCoDt8+fe38QKIteUcePGLa6urj4U8ugQQg4fPtztyrFrSUxMJEajMeLk6Vri4+NJbGxsWG0WLlxIqqurwxky/sCBA/9rMBgWhSip+pxUA0YsEqRzcRx3z44dO/7b5/OFpHcRYXr87LPPyA033BD2w4yJiSHJyckRmS5VKhVJTU2VtZRLlblz55JvvvkmHEIRl8tl++CDD94MgVALAdzU19NfRIgVRK7xABa++OKLy1paWmrDehV5nhQXF4ctwYKL2WwmaWlpJC4urlsTRzhFoVCQxMTEHhMJwqLkrrvuCsl63hWXLl0qmzdv3uMhkqrPVn9SpTf+WD0GwzCZAG7Mzs7Wbdq0afGECRPmMWHmrC0pKcHKlSuxefPmPvGj12q10Ol0UCqV4DiuY4D8joo8z4NlWRBCwPM83G43bDZbr07R+GE0GnH//ffjueee684ZTwSfz+f68ssv/98PfvCDLW63u7uHSQAcI4RIpmvuMwykxOoivZIFl4yFr7zyystWq/Vi2K8pIaS+vp78/ve/J/n5+QM+1fWmMAxDCgsLyYcffkhsNltPfjq5fPlyxYMPPvh0iFJqQV9Y1KNaYvnBMIweQAEAc2JionL9+vU/mDZt2kKFQtGjcDVlZWX45JNPsHHjRpw8KZmROGJgWRaTJk3CokWLcO+998qma5ODw+Fo3LRp098efvjhfTJHtILRBOAbQkjfpJQNARElFtrJxQG4AUAmAMyePTtu5cqV944bN+72cKfHYFRWVmL79u3Yvn07SkpKcPXq1T7td6hIT0/H9OnTMX/+fMydO7dXuQl5nvceO3Zs+/333/8/p06dCtWnuxJAKSFEfMa+HxFxYvnBMMwwYaWiBIAlS5akL1++fFFOTs7UsJLaSODkyZP4+uuvcfToUZSVlaG0tLTPQ1umpqZi/PjxuOGGGzBx4kTcdtttfZLihOd53+nTp7/6+c9/vr64uFicn4QON4AjhBDJfN/9iaghFtrJpQVwM4COjAT/9V//NeqZZ565Nz09fWJfECwYDQ0NqKqqQmVlJWpqamCxWNDQ0IDGxka0tbWJUhTrdDrodDrEx8d3lIyMDGRlZSEzMxMmk6kvuwee5z0nTpzYV1RUtFEmPhUN9QAO+9OPRAJRRSw/BOk1EUBHfJ3HHntsxPPPP39XXl7eNJZlr+nAWx6Px/7tt99+8corr/xLIjyjZFMA5QOy6usGUUkstJNLLdi8Omm4hYWF5tdee23m1KlTbzcajWmR62Hf4+rVq6d2796946WXXiqRifJCAwFQLZAqKmIYRC2x/GAYJkZQ7jvFnuQ4jikqKsq75557po8ePbpApVJJ5+iIYrS2tl4pKysr+fDDD/eupmd86A5WAN8RQqIqeWTUE8sPhmGGA8gDICKQ2WzmioqKJsybN29Kdnb2TTqdLqoDoDY1NdWcPn36cHFx8f4VK1acDdFk0BUtQtSXiCjn3WHQEAvt5GIAjACQC8Agdd8TTzyRcc8999yYm5ubn5KSkqtSqSTvHQi0tbU11NTUnCwvL69Yu3bt0S1btvTG9mEDcFKIVhy1D29QEcsPgWCpAEZ1nSK7guM45qGHHho2d+7cUSNHjsxITU3NTEhIyFCr1X27hBPQ2tp6xWKxXKitrb1w4sSJyo0bN57eunUrLcVtuGgQYn7WRTOh/BiUxAoGwzBxALIADKNkjJVEfn6+bsaMGUnjx49PysjISDKZTEaj0Wg0GAwmjUajBwD/v2jf+uJdLpeDEELa2tpsra2tNpvNZrNarS2VlZX1x44du7Jt27b6+vr63m8eBuAPzl8pFZ04WjHoieUHwzBKAMOFVWRU61jdgAh5lS9IBecfDLhmiBUMwdA6HEAagLi+Nqz2A4iQ/b0WQG3XhEeDEdcksYLBMIxKsOQnA0gAoA+h2UCgFcBVwUp+JTiJ5LWAa55YXcEwjEaYKuMAmAGYAPR3SGcngGahWIU4nhHbbhkIXHfEokGQakZBmumEoha2lFTC/xmh+BcIXmEK44UNX39xAWgTJFIbAFu0WMMHEv8fjXXJykHzTbkAAAAASUVORK5CYII=",
 "logo.png": "...",
 "changelog.md": "IyB2MC4yLjgKCi0gdGhlIHBhY2thZ2Ugbm93IHNoaXBzIGBhYm91dC5tZGAsIGBjaGFuZ2Vsb2cubWRgIGFuZCBgc3VwcG9ydC5tZGAKLSBidWlsZHMgcnVuIG9uIEdpdEh1YiBBY3Rpb25zIGFuZCByZWxlYXNlcyBsaXZlIG9uIEdpdEh1YgotIG1vZCBsaW5rcyBwb2ludCBhdCB0aGlzIHJlcG8KCiMgdjAuMi43CgotIHBpY2tpbmcgYSBub25nIG5vdyByZWZyZXNoZXMgdGhhdCByb3cncyBuYW1lIGFuZCBhcnRpc3QgcmlnaHQgYXdheSwgaW5zdGVhZCBvZiB3YWl0aW5nIGZvciBhIHNjcm9sbAoKIyB2MC4yLjYKCi0gZml4ZWQgdGhlIG5hbWUvYXJ0aXN0IHBsYWNlbWVudCBmb3IgcmVhbCAtIHRoZSBsYWJlbHMgd2VyZSBhIGZ1bGwgcm93IHRvbyBoaWdoIChyb3cgc3BhY2UgaXMgY2VudGVyZWQgb24gdGhlIHJvdydzIG1pZGRsZSwgbm90IGJvdHRvbS1sZWZ0KQotIG5hbWUgc2l0cyBqdXN0IGFib3ZlIHRoZSByb3cncyBtaWRkbGUsIGFydGlzdCBqdXN0IGJlbG93IGl0IC0gc2FtZSBmZWVsIGFzIHRoZSBvcmlnaW5hbCBicm93c2VyCgojIHYwLjIuNQoKLSB0aWdodGVyIHRleHQgcGFzczogbmFtZSBtYXggc2l6ZSAwLjU1LCB0ZXh0IGh1Z2dpbmcgdGhlIHJvdydzIGxlZnQgZWRnZQoKIyB2MC4yLjQKCi0gbG9uZyB0aXRsZXMgc2hyaW5rIHRvIGZpdCBpbnN0ZWFkIG9mIGdldHRpbmcgY3V0IHNob3J0OyAiLi4uIiBpcyBvbmx5IGZvciB0cnVseSBhYnN1cmQgbmFtZXMgbm93Ci0gdGhlIHNlYXJjaCBiYXIgcmVuZGVycyBhYm92ZSB0aGUgcm93cywgc28gc2Nyb2xsaW5nIHJvd3MgcGFzcyBiZWhpbmQgaXQKCiMgdjAuMi4zCgotIHRoZSBkaXNjIG9uIGVhY2ggcm93IG5vdyBvcGVucyBKdWtlYm94J3Mgbm9uZyBwaWNrZXIgZm9yIHRoYXQgc29uZwoKIyB2MC4yLjIKCi0gbGl2ZSBpbi11c2Ugc3RhdGUsIHJlY3Vyc2l2ZSBOT05HIGxvb2t1cCwgc2Nyb2xsaW5nL3NwYWNpbmcgcG9saXNoCi0gZmlsdGVyIGJ1dHRvbnMgZHJvcHBlZCAtIHNlYXJjaGluZyBjb3ZlcnMgaXQKCiMgdjAuMi4xCgotIGZpeGVkIGZpbHRlciBpY29ucywgYSBOT05HIHBpY2t1cCByYWNlLCBhbmQgc2Nyb2xsaW5nIGlzIGVhc2VkIG5vdwoKIyB2MC4yLjAKCi0gZmlyc3QgR2VvZGUgYnVpbGQ6IGZ1enp5IHNlYXJjaCBhY3Jvc3MgbmFtZSArIGFydGlzdCwgcHJldmlldywgZGVsZXRlLCBzZWxlY3QsIHNldHRpbmdzCg==",
 "support.md": "IyB0aGFua3MgZm9yIHVzaW5nIHNvbmcgc2VhcmNoIQoKaWYgeW91IGxpa2UgdGhlIG1vZCBhbmQgd2FudCB0byB0aHJvdyBhIGNvaW4gbXkgd2F5OgoKKipba28tZmkuY29tL25vdGxvdXN5XShodHRwczovL2tvLWZpLmNvbS9ub3Rsb3VzeSkqKgoKdGhpcyBpcyBhIHBvcnQgb2YgbWF0Y29vbCdzIFtzbWFsbC1nZC1tb2RzXShodHRwczovL2dpdGh1Yi5jb20vbWF0Y29vbC9zbWFsbC1nZC1tb2RzKQpzb25nLXNlYXJjaCwgc28gbW9zdCBvZiB0aGUgY3JlZGl0IGdvZXMgdG8gaGltIC0gY29uc2lkZXIgc3RhcnJpbmcgaGlzIHJlcG8gaW5zdGVhZCA6KQoKbm8gcHJlc3N1cmUgYXQgYWxsIC0gdXNpbmcgdGhlIG1vZCBhbmQgcmVwb3J0aW5nIGJ1Z3MgaXMgYWxyZWFkeSBwbGVudHkuCg==",
 "LICENSE": "TUlUIExpY2Vuc2UKCkNvcHlyaWdodCAoYykgMjAyNiBub3Rsb3VzeWJvb2sKClRoaXMgcHJvamVjdCBpcyBhIEdlb2RlIHBvcnQgb2YgInNvbmctc2VhcmNoIiBmcm9tIHNtYWxsLWdkLW1vZHMKKGh0dHBzOi8vZ2l0aHViLmNvbS9tYXRjb29sL3NtYWxsLWdkLW1vZHMpLCBDb3B5cmlnaHQgKGMpIDIwMjEgbWF0LCB3aGljaCBpcwphbHNvIGxpY2Vuc2VkIHVuZGVyIHRoZSBNSVQgTGljZW5zZS4gSXRzIG5vdGljZSBpcyBwcmVzZXJ2ZWQgYmVsb3cgYXMKcmVxdWlyZWQuCgpUaGUgYnVuZGxlZCBOT05HIHBpbi1kaXNjIGFydHdvcmsgKHJlc291cmNlcy9ub25nX2Rpc2MucG5nKSBjb21lcyBmcm9tCkp1a2Vib3ggKGh0dHBzOi8vZ2l0aHViLmNvbS9GbGVleW0vanVrZWJveCksIE1JVCBsaWNlbnNlZC4KCi0tLS0KCkNvcHlyaWdodCAoYykgMjAyMSBtYXQgKHNtYWxsLWdkLW1vZHMpCgpQZXJtaXNzaW9uIGlzIGhlcmVieSBncmFudGVkLCBmcmVlIG9mIGNoYXJnZSwgdG8gYW55IHBlcnNvbiBvYnRhaW5pbmcgYSBjb3B5Cm9mIHRoaXMgc29mdHdhcmUgYW5kIGFzc29jaWF0ZWQgZG9jdW1lbnRhdGlvbiBmaWxlcyAodGhlICJTb2Z0d2FyZSIpLCB0byBkZWFsCmluIHRoZSBTb2Z0d2FyZSB3aXRob3V0IHJlc3RyaWN0aW9uLCBpbmNsdWRpbmcgd2l0aG91dCBsaW1pdGF0aW9uIHRoZSByaWdodHMKdG8gdXNlLCBjb3B5LCBtb2RpZnksIG1lcmdlLCBwdWJsaXNoLCBkaXN0cmlidXRlLCBzdWJsaWNlbnNlLCBhbmQvb3Igc2VsbApjb3BpZXMgb2YgdGhlIFNvZnR3YXJlLCBhbmQgdG8gcGVybWl0IHBlcnNvbnMgdG8gd2hvbSB0aGUgU29mdHdhcmUgaXMKZnVybmlzaGVkIHRvIGRvIHNvLCBzdWJqZWN0IHRvIHRoZSBmb2xsb3dpbmcgY29uZGl0aW9uczoKClRoZSBhYm92ZSBjb3B5cmlnaHQgbm90aWNlIGFuZCB0aGlzIHBlcm1pc3Npb24gbm90aWNlIHNoYWxsIGJlIGluY2x1ZGVkIGluIGFsbApjb3BpZXMgb3Igc3Vic3RhbnRpYWwgcG9ydGlvbnMgb2YgdGhlIFNvZnR3YXJlLgoKVEhFIFNPRlRXQVJFIElTIFBST1ZJREVEICJBUyBJUyIsIFdJVEhPVVQgV0FSUkFOVFkgT0YgQU5ZIEtJTkQsIEVYUFJFU1MgT1IKSU1QTElFRCwgSU5DTFVESU5HIEJVVCBOT1QgTElNSVRFRCBUTyBUSEUgV0FSUkFOVElFUyBPRiBNRVJDSEFOVEFCSUxJVFksCkZJVE5FU1MgRk9SIEEgUEFSVElDVUxBUiBQVVJQT1NFIEFORCBOT05JTkZSSU5HRU1FTlQuIElOIE5PIEVWRU5UIFNIQUxMIFRIRQpBVVRIT1JTIE9SIENPUFlSSUdIVCBIT0xERVJTIEJFIExJQUJMRSBGT1IgQU5ZIENMQUlNLCBEQU1BR0VTIE9SIE9USEVSCkxJQUJJTElUWSwgV0hFVEhFUiBJTiBBTiBBQ1RJT04gT0YgQ09OVFJBQ1QsIFRPUlQgT1IgT1RIRVJXSVNFLCBBUklTSU5HIEZST00sCk9VVCBPRiBPUiBJTiBDT05ORUNUSU9OIFdJVEggVEhFIFNPRlRXQVJFIE9SIFRIRSBVU0UgT1IgT1RIRVIgREVBTElOR1MgSU4gVEhFClNPRlRXQVJFLgo=",
}
ROOT = pathlib.Path('/kaggle/working/song-search')
for rel, b64data in FILES.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64data))
    print('wrote', p, p.stat().st_size, 'bytes')

# the SDK requires mod.json "geode" to match its own version exactly, and
# compares major.minor with the "v" prefix stripped from both sides, so the
# field must carry no prefix
sdk_ver = pathlib.Path('/tmp/work/sdk/VERSION').read_text().strip().lstrip('v')
mj = ROOT / 'mod.json'
m = json.loads(mj.read_text())
if m.get('geode') != sdk_ver:
    print('mod.json geode', m.get('geode'), '->', sdk_ver)
    m['geode'] = sdk_ver
    mj.write_text(json.dumps(m, indent=2) + chr(10))
print('mod.json geode =', m['geode'])

In [ ]:
%%bash

set -eo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "4. mod source + native logic test"
cd /kaggle/working/song-search || exit 10
find . -type f | sort | tee -a "$LOG"
g++ -std=c++20 -O2 -Wall -I src -o /tmp/work/logic_test tests/logic_test.cpp
/tmp/work/logic_test | tee -a "$LOG"
log "logic test rc=$?"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "5. configure (cross-compile -> Win64)"
cd /kaggle/working/song-search || exit 11
# capture-orig-geode: remember the mod's declared minimum Geode version
# before the Geode packaging step rewrites it to the SDK version.
python3 -c "import json;print(json.load(open('mod.json'))['geode'])" > /tmp/work/orig_mod_geode.txt
echo "source mod.json geode: $(cat /tmp/work/orig_mod_geode.txt)" | tee -a "$LOG"
export PATH=/usr/local/bin:/tmp/work/geode-cli:/tmp/work/xwin:$PATH
export SPLAT_DIR=/tmp/work/splat
export TOOLCHAIN=/tmp/work/toolchain/clang-cl-msvc.cmake
export HOST_ARCH=x86_64
export GEODE_SDK=/tmp/work/sdk
export GEODE_BINDINGS_REPO_PATH=/tmp/work/bindings

rm -rf build
cmake \
  -DCMAKE_TOOLCHAIN_FILE=$TOOLCHAIN \
  -DCMAKE_BUILD_TYPE=Release \
  -DGEODE_TARGET_PLATFORM=Win64 \
  -G Ninja \
  -B build > /tmp/work/configure.log 2>&1
RC=$?
tail -50 /tmp/work/configure.log | tee -a "$LOG"
echo "configure rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 12

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "6. build"
cd /kaggle/working/song-search || exit 13
export PATH=/usr/local/bin:/tmp/work/geode-cli:$PATH
export GEODE_SDK=/tmp/work/sdk
time cmake --build build --config Release > /tmp/work/build.log 2>&1
RC=$?
tail -70 /tmp/work/build.log | tee -a "$LOG"
echo "build rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 14
# restore-orig-geode: Geode's packaging rewrites mod.json's `geode` field to
# the SDK version, which makes the mod refuse to load on older loaders.
# Set it back to the version the mod actually targets.
python3 - <<'PY' | tee -a "$LOG"
import json, re, glob, os, shutil, zipfile
want = "5.11.1"
t = '/kaggle/working/song-search/mod.json'
txt = open(t, encoding='utf-8').read()
new = re.sub(r'"geode"\s*:\s*"[^"]+"', '"geode": "%s"' % want, txt, count=1)
assert new != txt or ('"geode": "%s"' % want) in txt, 'tree mod.json geode not found'
json.loads(new)
open(t, 'w', encoding='utf-8').write(new)
geos = glob.glob('/kaggle/working/song-search/build/**/*.geode', recursive=True)
assert geos, 'no .geode in build/'
g = max(geos, key=os.path.getmtime)
zin = zipfile.ZipFile(g)
items = []
for info in zin.infolist():
    d = zin.read(info.filename)
    if info.filename == 'mod.json':
        s = d.decode('utf-8')
        s2 = re.sub(r'"geode"\s*:\s*"[^"]+"', '"geode": "%s"' % want, s, count=1)
        assert s2 != s or ('"geode": "%s"' % want) in s, 'zip mod.json geode not found'
        json.loads(s2)
        d = s2.encode('utf-8')
    items.append((info, d))
zin.close()
if 'logo.png' not in [i.filename for i, _ in items]:
    lp = '/kaggle/working/song-search/logo.png'
    if os.path.exists(lp):
        zi = zipfile.ZipInfo('logo.png')
        zi.compress_type = zipfile.ZIP_DEFLATED
        items.append((zi, open(lp, 'rb').read()))
        print('logo.png appended to package')
tmp = g + '.tmp'
with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zo:
    for info, d in items:
        zo.writestr(info, d)
shutil.move(tmp, g)
m = json.loads(zipfile.ZipFile(g).read('mod.json').decode('utf-8'))
print('restored mod geode:', m['geode'], '| version:', m['version'], '| file:', g)
PY
log "BUILD DONE"
find build -maxdepth 3 -type f \( -name '*.geode' -o -name '*.dll' -o -name '*.lib' \) \
  -exec ls -la {} \; | tee -a "$LOG"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
cd /kaggle/working/song-search || exit 15
GEO=$(find build -maxdepth 3 -name '*.geode' 2>/dev/null | head -1)
if [ -z "$GEO" ]; then
  echo "NO .geode PRODUCED" | tee -a "$LOG"
  find build -maxdepth 3 -type f 2>/dev/null | head -30 | tee -a "$LOG"
  exit 16
fi
echo "geode: $GEO ($(stat -c%s "$GEO") bytes)" | tee -a "$LOG"
unzip -l "$GEO" | tee -a "$LOG"

log "7. DLL architecture check"
mkdir -p /tmp/work/geocheck
cd /tmp/work/geocheck
rm -rf ./* 2>/dev/null || true
unzip -o -q "/kaggle/working/song-search/$GEO"
DLL=$(find . -name '*.dll' | head -1)
echo "mod dll: $DLL" | tee -a "$LOG"
python3 - "$DLL" <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import sys, struct
def pe_info(p):
    d = open(p,'rb').read()
    assert d[:2]==b'MZ'
    e = struct.unpack_from('<I', d, 0x3c)[0]
    assert d[e:e+4]==b'PE\x00\x00'
    mach  = struct.unpack_from('<H', d, e+4)[0]
    chars = struct.unpack_from('<H', d, e+22)[0]
    magic = struct.unpack_from('<H', d, e+24)[0]
    return dict(
      machine={0x14c:'i386 (32-bit)',0x8664:'x86_64 (64-bit)',0xaa64:'arm64'}.get(mach,hex(mach)),
      is_dll=bool(chars & 0x2000),
      pe_format='PE32+' if magic==0x20b else 'PE32',
      size=len(d))
i = pe_info(sys.argv[1])
for k,v in i.items(): print('  %s: %s' % (k,v))
ok = i['machine'].startswith('x86_64') and i['is_dll'] and i['pe_format']=='PE32+'
print('  ARCH MATCHES GD 2.2081 (x86_64):', 'YES' if ok else 'NO')
PY

log "8. mod.json sanity"
python3 - <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import json
m = json.load(open('/kaggle/working/song-search/mod.json'))
print('id     :', m['id'])
print('version:', m['version'])
print('geode  :', m['geode'])
print('gd.win :', m['gd']['win'])
assert m['gd']['win'] in ('*', '2.2081')
print('mod.json OK')
PY

# --- keep the kernel output small ---
# `kaggle kernels output` downloads everything under /kaggle/working, and the
# build tree holds ~30MB of static libs. Copy the deliverables out and drop
# the build tree so the package can actually be fetched.
cd /kaggle/working/song-search
log "9. pruning build tree / staging output"
if [ -f build/matcool.song-search.geode ]; then
  cp -f build/matcool.song-search.geode /kaggle/working/matcool.song-search.geode
  cp -f build/matcool.song-search.dll   /kaggle/working/matcool.song-search.dll 2>/dev/null || true
fi
cp -f /kaggle/working/BUILD-LOG.txt /kaggle/working/song-search/BUILD-LOG.txt 2>/dev/null || true
rm -rf build/_deps
rm -rf build/bindings
rm -f  build/bindings/*.lib build/*.exp build/*.ilk build/CMakeFiles/*/*.obj 2>/dev/null || true
du -sh /kaggle/working/song-search /kaggle/working/*.geode 2>/dev/null | tee -a "$LOG"
log "ALL DONE"
